### Load all files anyways + in case of vis

In [ ]:
import pandas as pd

files = {
    '12a macro base':       'data/individual_merges/parquet_saves/macro_ind_base_me.parquet',
    '12b macro robust':     'data/individual_merges/parquet_saves/macro_ind_robust_me.parquet',
    '11a base lag1':  'data/engineered/parquet_saves/me_merged_base.parquet',
    '11b robust lag1': 'data/engineered/parquet_saves/me_merged_robust.parquet',
    '11c base lag2':  'data/engineered/parquet_saves/me_merged_base_lag2.parquet',
    '11d robust lag2': 'data/engineered/parquet_saves/me_merged_robust_lag2.parquet',
}

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pickle
import warnings
from pathlib import Path
from scipy import stats

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 150, 'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

# Micro/macro membership is decided by EXCLUSION from this explicit macro list, mirroring how
# the modelling notebooks define the blocks (NB11: `micro_cols = [c for c in df.columns if c
# not in id_cols + [target] + macro_cols]`). Do NOT reintroduce a prefix-based test: the micro
# codes 'inc_nii', 'inc_op' and 'cr_impchg' match none of the g_/l_/r_/c_/x_/p_ prefixes, so a
# prefix rule silently counts three micro features as macro - it previously did, in both the
# block aggregates and the bar colours. Union across all four specs, so one set serves 11a-11d.
MACRO_FEATURES = {
    'NGDP_RPCH', 'NGDPD', 'PCPIPCH', 'LUR', 'BCA_NGDPD', 'BIS_REER', 'RXF11FX_REVS',
    'reer_g', 'fx_g', 'FS.AST.PRVT.GD.ZS', 'FM.AST.NFRG.CN', 'nfa_gdp', 'credit_g',
    'credit_gap', 'credit_gap_missing',  # A23/A24
}

In [ ]:
BASE = Path("..").resolve()
OUTPUT = BASE / 'outputs/figures' / 'results_11a'
OUTPUT.mkdir(exist_ok=True)

# Results excels — highest draft number per spec is the most recent run
RESULT_FILES = {
    '11a_base_lag1':          BASE / 'outputs/results/11a/results_11a_final.xlsx',
    '11b_robust_lag1':        BASE / 'outputs/results/11b/results_11b_final.xlsx',
    '11c_base_lag2':          BASE / 'outputs/results/11c/results_11c_final.xlsx',  # A10
    '11d_robust_lag2':        BASE / 'outputs/results/11d/results_11d_final.xlsx',
    'rt2a_base_lag2_fixed':   BASE / 'outputs/results/rt2a/results_rt2a_final.xlsx',
    'rt2b_robust_lag2_fixed': BASE / 'outputs/results/rt2b/results_rt2b_final.xlsx',
    '12a_macro_base':         BASE / 'outputs/results/12a/results_12a_final.xlsx',
    '12b_macro_robust':       BASE / 'outputs/results/12b/results_12b_final.xlsx',
}

# Proba stores for DeLong tests
PROBA_FILES = {
    '11a_base_lag1':          BASE / 'outputs/results/11a/proba_store_final.pkl',
    '11b_robust_lag1':        BASE / 'outputs/results/11b/proba_store_final.pkl',
    '11c_base_lag2':          BASE / 'outputs/results/11c/proba_store_final.pkl',  # A10
    '11d_robust_lag2':        BASE / 'outputs/results/11d/proba_store_final.pkl',
    'rt2a_base_lag2_fixed':   BASE / 'outputs/results/rt2a/proba_store_final.pkl',
    'rt2b_robust_lag2_fixed': BASE / 'outputs/results/rt2b/proba_store_final.pkl',
    '12a_macro_base':         BASE / 'outputs/results/12a/proba_store_final.pkl',
    '12b_macro_robust':       BASE / 'outputs/results/12b/proba_store_final.pkl',
}

# SHAP parquets — NB13b and 13d were rerun after the FM.AST.NFRG.CN fix (variable absent from outputs, checked 17-09-2026)
SHAP_FILES = {
    '11a': BASE / 'outputs/attribution/outputs_11a/shap_values_11a.parquet',
    '11b': BASE / 'outputs/attribution/outputs_11b/shap_values_11b.parquet',
    '11c': BASE / 'outputs/attribution/outputs_11c/shap_values_11c.parquet',
    '11d': BASE / 'outputs/attribution/outputs_11d/shap_values_11d.parquet',
}

# Main results - 11a

## NB11a — Load and merge metrics

In [ ]:
EXCEL_11A = RESULT_FILES['11a_base_lag1']

# mean_metrics_baseline: pre-aggregated in NB11a — has auroc, f1, recall, precision,
# miss_rate, false_alarm_rate, loss_08 but is missing auprc, brier, best_threshold, loss_07/09
mean_base = pd.read_excel(EXCEL_11A, sheet_name='mean_metrics_baseline', index_col=[0, 1])
mean_base = mean_base.reset_index()

# Exclude lr_no_dummies — it is included in the NB11a Excel as a diagnostic but is not a main model
mean_base = mean_base[mean_base['model'] != 'lr_no_dummies']

# all_results: one row per (model, dataset, test_year, variant) — contains all metrics including auprc
all_results = pd.read_excel(EXCEL_11A, sheet_name='all_results')

# Filter to baseline_t1 only (main results, not RT1) and drop lr_no_dummies
baseline = all_results[
    (all_results['variant'] == 'baseline_t1') &
    (all_results['model'] != 'lr_no_dummies')
]

# Average the metrics that are missing from mean_metrics_baseline across the three test years
# best_threshold is also here for the cut-off table. A39: the Brier score is no longer reported;
# the models are class-weighted and not calibrated, so it is not a probability score.
extra = (
    baseline
    .groupby(['model', 'dataset'])[['auprc', 'best_threshold', 'loss_07', 'loss_09']]
    .mean()
    .round(3)
    .reset_index()
)

# Merge into one complete metrics frame
metrics = mean_base.merge(extra, on=['model', 'dataset'], how='left')

print(f"Rows: {len(metrics)}  |  Columns: {metrics.columns.tolist()}")
metrics

In [ ]:
# Display ordering and labels used across all tables
MODEL_ORDER = ['logistic_regression', 'decision_tree', 'random_forest', 'xgboost', 'lightgbm', 'mlp']
MODEL_LABELS = {
    'logistic_regression': 'Logistic Regression',
    'decision_tree':       'Decision Tree',
    'random_forest':       'Random Forest',
    'xgboost':             'XGBoost',
    'lightgbm':            'LightGBM',
    'mlp':                 'MLP',
}
DATASET_ORDER  = ['micro', 'macro', 'integrated']
DATASET_LABELS = {'micro': 'Micro-only', 'macro': 'Macro-only', 'integrated': 'Integrated'}

# Apply ordering so all tables sort consistently
m = metrics.copy()
m['model']   = pd.Categorical(m['model'],   categories=MODEL_ORDER,   ordered=True)
m['dataset'] = pd.Categorical(m['dataset'], categories=DATASET_ORDER, ordered=True)
m = m.sort_values(['model', 'dataset']).reset_index(drop=True)

## Table A — AUROC + AUPRC

In [ ]:
# Wide format: rows = model, column groups = dataset, values = auroc + auprc
table_a = m.pivot(index='model', columns='dataset', values=['auroc', 'auprc'])

# Flatten MultiIndex columns: (auroc, micro) → "AUROC — Micro-only"
table_a.columns = [
    f"{metric.upper()} — {DATASET_LABELS[ds]}"
    for metric, ds in table_a.columns
]

# Reorder columns so each dataset shows AUROC then AUPRC side by side
col_order = [
    f"{metric} — {DATASET_LABELS[ds]}"
    for ds in DATASET_ORDER
    for metric in ['AUROC', 'AUPRC']
]
table_a = table_a[col_order].round(3)
table_a.index = table_a.index.map(MODEL_LABELS)
table_a.index.name = 'Model'

table_a

## Table B — F1 + Loss functions (tall format)

In [ ]:
# Tall format: config-first sort (all micro → all macro → all integrated) so the reader
# can compare the error profile across models within each information type, then across types.
# This aligns with the thesis question: does integrated change the miss/false alarm balance?
#
# Columns:
#   Miss Rate + False Alarm Rate: the two components of L(λ) = λ|FNR + (1-λ)|FPR
#   L(0.8): primary asymmetric loss metric (moderate FN penalty)
#   ΔL(0.9): signed delta L(0.9) − L(0.8), shows direction when FN penalty increases
#             without adding a full redundant column; negative = further improvement

table_b = (
    m[['model', 'dataset', 'miss_rate', 'false_alarm_rate', 'loss_08', 'loss_09']]
    .copy()
    .sort_values(['dataset', 'model'])  # config-first; Categorical ordering keeps micro→macro→integrated
    .reset_index(drop=True)
)

# ΔL(0.9): negative means FPR > FNR at threshold — model flags more than it misses,
# so increasing FN weight further lowers total loss
table_b['delta_l09'] = (table_b['loss_09'] - table_b['loss_08']).round(3).apply(lambda x: f'{x:+.3f}')
table_b = table_b.drop(columns=['loss_09'])

table_b['model']   = table_b['model'].map(MODEL_LABELS)
table_b['dataset'] = table_b['dataset'].map(DATASET_LABELS)

table_b = table_b.rename(columns={
    'model':            'Model',
    'dataset':          'Configuration',
    'miss_rate':        'Miss Rate',
    'false_alarm_rate': 'False Alarm Rate',
    'loss_08':          'L(0.8)',
    'delta_l09':        'ΔL(0.9)',
})

table_b

## Alarm cut-off (A1: chosen on the previous year's out-of-sample predictions)

In [ ]:
calibration = (
    m[['model', 'dataset', 'best_threshold']]
    .copy()
    .sort_values(['dataset', 'model'])
    .reset_index(drop=True)
)
calibration['model']   = calibration['model'].map(MODEL_LABELS)
calibration['dataset'] = calibration['dataset'].map(DATASET_LABELS)
calibration = calibration.rename(columns={
    'model': 'Model', 'dataset': 'Configuration',
    'best_threshold': 'Alarm cut-off (mean over test years)'
})

calibration

## Export to Excel

In [ ]:
OUT = BASE / 'outputs/tables'
OUT.mkdir(exist_ok=True)

with pd.ExcelWriter(OUT / 'results_15_11a.xlsx', engine='openpyxl') as writer:
    table_a.to_excel(writer, sheet_name='table_a_auroc_auprc')
    table_b.to_excel(writer, sheet_name='table_b_f1_loss', index=False)
    calibration.to_excel(writer, sheet_name='calibration', index=False)
    metrics.to_excel(writer, sheet_name='full_metrics', index=False)

print(f"Saved → {OUT / 'results_15_11a.xlsx'}")

## ROC and PR Panel Figures

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

# Load predicted probabilities from NB11a
# Keys: (model, dataset, test_year, variant) → (y_true, y_proba)
with open(PROBA_FILES['11a_base_lag1'], 'rb') as f:
    proba_store = pickle.load(f)

print(f"Loaded {len(proba_store)} entries")
print("Sample keys:", list(proba_store.keys())[:4])

In [ ]:
DATASET_COLORS = {
    'micro':      '#1f77b4',  # blue
    'macro':      '#ff7f0e',  # orange
    'integrated': '#2ca02c',  # green
}
TEST_YEARS = [2010, 2011, 2012]
VARIANT    = 'baseline_t1'
# A55: the curves pool the test years, and the AUROC of a pooled curve differs from the mean of the
# per-year AUROCs in Table A. The legends show the Table A value so the two never disagree.
TABLE_AUROC = {(str(r.model), str(r.dataset)): r.auroc for r in m.itertuples()}
TABLE_AUPRC = {(str(r.model), str(r.dataset)): r.auprc for r in m.itertuples()}

# --- ROC Panel ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, model in enumerate(MODEL_ORDER):
    ax = axes[i]

    for ds in DATASET_ORDER:
        # Pool y_true and y_proba across all test years — gives one curve per spec
        y_true_all, y_proba_all = [], []
        for yr in TEST_YEARS:
            key = (model, ds, yr, VARIANT)
            if key in proba_store:
                y_true, y_proba = proba_store[key]
                y_true_all.extend(y_true)
                y_proba_all.extend(y_proba)

        # Skip if no data or no positive class in pooled set (would break roc_curve)
        if not y_true_all or len(set(y_true_all)) < 2:
            continue

        fpr, tpr, _ = roc_curve(y_true_all, y_proba_all)
        auroc_val   = TABLE_AUROC[(model, ds)]  # A55: mean of per-year AUROC, as in Table A

        ax.plot(fpr, tpr, color=DATASET_COLORS[ds], lw=2,
                label=f"{DATASET_LABELS[ds]} ({auroc_val:.3f})")

    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)  # chance level
    ax.set_title(MODEL_LABELS[model], fontsize=10, fontweight='bold')
    ax.set_xlabel('False Positive Rate', fontsize=9)
    ax.set_ylabel('True Positive Rate', fontsize=9)
    ax.legend(fontsize=8, loc='lower right', title='Config (AUROC)', title_fontsize=8)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

#fig.suptitle('ROC Curves — Baseline Lag-1 | Pooled Test Years 2010–2012',
#             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT / 'roc_panel_11a.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → roc_panel_11a.png")
# -- Split into two vertical 3x1 figures for document pasting --
for part_models, suffix in [
    (MODEL_ORDER[:3], 'p1'),
    (MODEL_ORDER[3:], 'p2'),
]:
    fig2, axes2 = plt.subplots(3, 1, figsize=(7, 15))
    for ax_i, model in enumerate(part_models):
        ax = axes2[ax_i]
        for ds in DATASET_ORDER:
            y_true_all, y_proba_all = [], []
            for yr in TEST_YEARS:
                key = (model, ds, yr, VARIANT)
                if key in proba_store:
                    y_true, y_proba = proba_store[key]
                    y_true_all.extend(y_true)
                    y_proba_all.extend(y_proba)
            if not y_true_all or len(set(y_true_all)) < 2:
                continue
            fpr, tpr, _ = roc_curve(y_true_all, y_proba_all)
            auroc_val   = TABLE_AUROC[(model, ds)]  # A55: mean of per-year AUROC, as in Table A
            ax.plot(fpr, tpr, color=DATASET_COLORS[ds], lw=2,
                    label=f"{DATASET_LABELS[ds]} ({auroc_val:.3f})")
        ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
        ax.set_title(MODEL_LABELS[model], fontsize=10, fontweight='bold')
        ax.set_xlabel('False Positive Rate', fontsize=9)
        ax.set_ylabel('True Positive Rate', fontsize=9)
        ax.legend(fontsize=8, loc='lower right', title='Config (AUROC)', title_fontsize=8)
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
    plt.tight_layout()
    out_name = f'roc_panel_11a_{suffix}.png'
    fig2.savefig(OUTPUT / out_name, bbox_inches='tight', dpi=200)
    plt.show()
    print(f'Saved --> {out_name}')

In [ ]:
# --- PR Panel ---
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, model in enumerate(MODEL_ORDER):
    ax = axes[i]

    for ds in DATASET_ORDER:
        y_true_all, y_proba_all = [], []
        for yr in TEST_YEARS:
            key = (model, ds, yr, VARIANT)
            if key in proba_store:
                y_true, y_proba = proba_store[key]
                y_true_all.extend(y_true)
                y_proba_all.extend(y_proba)

        if not y_true_all or len(set(y_true_all)) < 2:
            continue

        precision, recall, _ = precision_recall_curve(y_true_all, y_proba_all)
        ap  = TABLE_AUPRC[(model, ds)]  # A55: mean of per-year AUPRC, as in Table A

        ax.plot(recall, precision, color=DATASET_COLORS[ds], lw=2,
                label=f"{DATASET_LABELS[ds]} ({ap:.3f})")

    ax.set_title(MODEL_LABELS[model], fontsize=10, fontweight='bold')
    ax.set_xlabel('Recall', fontsize=9)
    ax.set_ylabel('Precision', fontsize=9)
    ax.legend(fontsize=8, loc='upper right', title='Config (AUPRC)', title_fontsize=8)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

#fig.suptitle('Precision-Recall Curves — Baseline Lag-1 | Pooled Test Years 2010–2012',
 #            fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT / 'pr_panel_11a.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → pr_panel_11a.png")

In [ ]:
from sklearn.metrics import accuracy_score

# Accuracy is threshold-dependent — computed per (model, dataset, test_year) using
# that year's best_threshold (same approach as miss_rate/false_alarm_rate in NB11a),
# then averaged across the 3 test years. Reported in appendix only; not in main tables
# because under ~86% class imbalance it is dominated by the non-crisis majority.

acc_records = []
for _, row in baseline.iterrows():
    key = (row['model'], row['dataset'], row['test_year'], row['variant'])
    if key not in proba_store:
        continue
    y_true, y_proba = proba_store[key]
    y_pred = (np.array(y_proba) >= row['best_threshold']).astype(int)
    acc_records.append({
        'model':   row['model'],
        'dataset': row['dataset'],
        'accuracy': accuracy_score(y_true, y_pred),
    })

acc_mean = (
    pd.DataFrame(acc_records)
    .groupby(['model', 'dataset'])['accuracy']
    .mean()
    .round(3)
    .reset_index()
)

# Merge accuracy into the full metrics frame
metrics_with_acc = metrics.merge(acc_mean, on=['model', 'dataset'], how='left')

with pd.ExcelWriter(OUT / 'results_15_11a.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    metrics_with_acc.to_excel(writer, sheet_name='full_metrics', index=False)

## DeLong Test — significance of integrated vs micro / macro

In [ ]:
# Load raw DeLong results (one row per model | comparison | test_year | variant)
delong_raw = pd.read_excel(EXCEL_11A, sheet_name='delong_raw')

# Focus on H1 comparisons (integrated vs micro, integrated vs macro) for baseline_t1 only
h1 = delong_raw[
    (delong_raw['comparison'].isin(['integrated_vs_micro', 'integrated_vs_macro'])) &
    (delong_raw['variant'] == 'baseline_t1')
]

# Summarise per (comparison, model): mean z-statistic and fraction of test years significant at p<0.05
# With 3 test years, pct_sig=1.0 means all 3 significant; 0.667 means 2/3; etc.
delong_summary = (
    h1.groupby(['comparison', 'model'])
    .agg(
        mean_z   = ('z',      'mean'),
        pct_sig  = ('sig_boot', 'mean'),  # A33: country-cluster bootstrap, not DeLong
        n        = ('z',      'count'),
    )
    .round(3)
    .reset_index()
)
delong_summary['model_label'] = delong_summary['model'].map(MODEL_LABELS)
print(delong_summary.to_string(index=False))

In [ ]:
# Build asterisk map for Table A — mark integrated AUROC where it significantly outperforms
# Threshold: pct_sig > 0.5, a strict majority of test years with bootstrap p < 0.05 (A33, A54).
# '>= 0.5' let 1 of 2 years count for the two-window specifications.
# positive mean_z = integrated AUROC > comparison AUROC
SIG_THRESHOLD = 0.5

def asterisk(row):
    sig   = row['pct_sig'] > SIG_THRESHOLD  # A54: strict majority (2 of 3, or 2 of 2)
    better = row['mean_z'] > 0  # positive z = integrated is better
    if sig and better:
        return '*' if row['pct_sig'] < 1.0 else '**'
    return ''

delong_summary['sig_marker'] = delong_summary.apply(asterisk, axis=1)

# Pivot to model | comparison for easy reading
asterisk_table = delong_summary.pivot(
    index='model_label', columns='comparison', values=['mean_z', 'pct_sig', 'sig_marker']
)
asterisk_table.index.name = 'Model'
asterisk_table.columns.names = [None, None]
print("\n=== Asterisk map for Table A (integrated AUROC column) ===")
print("* = significant in a strict majority of windows (country-cluster bootstrap p<0.05), ** = all windows, positive z = integrated better")
print(asterisk_table.to_string())

In [ ]:
# Export DeLong results to the existing Excel
with pd.ExcelWriter(OUT / 'results_15_11a.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    delong_summary.to_excel(writer, sheet_name='delong_summary', index=False)
    asterisk_table.to_excel(writer, sheet_name='delong_asterisks')

print(f"Saved DeLong sheets → {OUT / 'results_15_11a.xlsx'}")

# Robustness Specs

In [ ]:
# Robustness spec definitions: key → (xlsx path, variant filter, n_test_years)
# n_test_years is noted explicitly because pct_sig thresholds depend on it:
#   11b: n=3 (2010-2012)  | 11c/11d: n=2 (2011-2012, lag-2 drops 2010)
#   rt1 variants: n=2 (2011-2012; 2010 skipped per onset filter) for lag-1 specs,
#                 n=2 (2011-2012) for lag-2 specs
#   RT2a/b: n=1 (2007 only) - pct_sig is binary (0 or 1), DeLong illustrative only
ROBUST_SPECS = {
    '11b_robust_lag1':        (RESULT_FILES['11b_robust_lag1'],        'baseline_t1', 3),
    '11c_base_lag2':          (RESULT_FILES['11c_base_lag2'],          'baseline_t1', 2),
    '11d_robust_lag2':        (RESULT_FILES['11d_robust_lag2'],        'baseline_t1', 2),
    'rt1_base_lag1':          (RESULT_FILES['11a_base_lag1'],          'rt1',         2),
    'rt1_robust_lag1':        (RESULT_FILES['11b_robust_lag1'],        'rt1',         2),
    'rt1_base_lag2':          (RESULT_FILES['11c_base_lag2'],          'rt1',         2),
    'rt1_robust_lag2':        (RESULT_FILES['11d_robust_lag2'],        'rt1',         2),
    'rt2a_base_lag2_fixed':   (RESULT_FILES['rt2a_base_lag2_fixed'],   'rt2_2yr',     1),
    'rt2b_robust_lag2_fixed': (RESULT_FILES['rt2b_robust_lag2_fixed'], 'rt2_2yr',     1),
}

# Short column labels for comparison tables and figure axes
ROBUST_LABELS = {
    '11a_base_lag1':          'Baseline / Lag-1',
    '11b_robust_lag1':        'Baseline Extended / Lag-1',
    '11c_base_lag2':          'Baseline / Lag-2',
    '11d_robust_lag2':        'Baseline Extended / Lag-2',
    'rt1_base_lag1':          'Onset excl. / Lag-1',
    'rt1_robust_lag1':        'Onset excl. / Ext.',
    'rt1_base_lag2':          'Onset excl. / Lag-2',
    'rt1_robust_lag2':        'Onset excl. / Ext. Lag-2',
    'rt2a_base_lag2_fixed':   'Pre-crisis / Lag-2',
    'rt2b_robust_lag2_fixed': 'Pre-crisis / Extended',
}


def load_spec_years(xlsx_path, variant_filter):
    """Per-year rows of all_results for one variant, without lr_no_dummies (A53)."""
    all_res = pd.read_excel(xlsx_path, sheet_name='all_results')
    return all_res[
        (all_res['variant'] == variant_filter) &
        (all_res['model'] != 'lr_no_dummies')
    ]


def load_spec_metrics(xlsx_path, variant_filter, years=None):
    """Aggregate all_results for a given variant across test years.
    Uses all_results directly so it works for any variant name across all specs.
    years: restrict to these test years (A44: comparisons on shared test years only)."""
    filtered = load_spec_years(xlsx_path, variant_filter)
    if years is not None:
        filtered = filtered[filtered['test_year'].isin(years)]
    agg_cols = ['auroc', 'auprc', 'loss_07', 'loss_08', 'loss_09']
    agg = (
        filtered
        .groupby(['model', 'dataset'])[agg_cols]
        .mean()
        .round(3)
        .reset_index()
    )
    agg['model']   = pd.Categorical(agg['model'],   categories=MODEL_ORDER,   ordered=True)
    agg['dataset'] = pd.Categorical(agg['dataset'], categories=DATASET_ORDER, ordered=True)
    return agg.sort_values(['model', 'dataset']).reset_index(drop=True)


def load_spec_delong(xlsx_path, variant_filter):
    """Summarise DeLong H1 comparisons (integrated vs micro/macro) for a given variant."""
    raw = pd.read_excel(xlsx_path, sheet_name='delong_raw')
    # A33: bootstrap significance where it exists; rt2 is not rerun (A46) and only has DeLong
    sig_col = 'sig_boot' if 'sig_boot' in raw.columns else 'sig_05'
    h1 = raw[
        (raw['comparison'].isin(['integrated_vs_micro', 'integrated_vs_macro'])) &
        (raw['variant'] == variant_filter)
    ]
    summary = (
        h1.groupby(['comparison', 'model'])
        .agg(mean_z=('z', 'mean'), pct_sig=(sig_col, 'mean'), n=('z', 'count'))
        .round(3)
        .reset_index()
    )
    summary['model_label'] = summary['model'].map(MODEL_LABELS)
    return summary


# Load NB11a metrics from the already-computed `m` frame (baseline_t1, no lr_no_dummies)
robust_metrics = {
    '11a_base_lag1': m[['model', 'dataset', 'auroc', 'auprc', 'loss_07', 'loss_08', 'loss_09']].copy()
}

# Load all robustness specs
for spec_key, (path, variant, n_yrs) in ROBUST_SPECS.items():
    robust_metrics[spec_key] = load_spec_metrics(path, variant)
    print(f"Loaded {spec_key} (n_test_years={n_yrs}): {len(robust_metrics[spec_key])} rows")

ALL_SPEC_KEYS = ['11a_base_lag1'] + list(ROBUST_SPECS.keys())

# A53: a DIFFERENCE between specifications is taken over matched test years only: per model,
# configuration and year, both sides must have a value (after A1 the threshold metrics of RT1 2011
# are empty). Averaging each side over its own years mixed a specification effect with a year effect
# (mean AUROC 0.477 in 2010 against 0.764/0.799 in 2011/2012). Levels (the comparison tables) stay
# on each specification's own window. The lag figure (Figure 2) stays unmatched by decision (05-08-2026).
SPEC_YEARS = {'11a_base_lag1': load_spec_years(RESULT_FILES['11a_base_lag1'], 'baseline_t1')}
SPEC_YEARS.update({k: load_spec_years(path, variant) for k, (path, variant, _) in ROBUST_SPECS.items()})


def matched_delta(spec_key, metric, base_key='11a_base_lag1'):
    """Mean over matched test years of (spec - base), per (model, dataset); NaN without a shared year."""
    keys = ['model', 'dataset', 'test_year']
    both = SPEC_YEARS[spec_key][keys + [metric]].merge(
        SPEC_YEARS[base_key][keys + [metric]], on=keys, suffixes=('_spec', '_base')).dropna()
    both['delta'] = both[f'{metric}_spec'] - both[f'{metric}_base']
    out = both.groupby(['model', 'dataset'])['delta'].mean()
    out.index = pd.MultiIndex.from_tuples([(str(a), str(b)) for a, b in out.index], names=['model', 'dataset'])
    return out


# --- Comparison tables: rows = model | config, cols = specs ---
def build_metric_comparison(metric):
    """Build wide table: (model, config) rows | spec columns for a single metric."""
    frames = []
    for spec_key in ALL_SPEC_KEYS:
        df = robust_metrics[spec_key][['model', 'dataset', metric]].copy()
        df = df.rename(columns={metric: ROBUST_LABELS[spec_key]})
        frames.append(df.set_index(['model', 'dataset']))
    combined = pd.concat(frames, axis=1)
    combined.index = pd.MultiIndex.from_tuples(
        [(MODEL_LABELS[str(m_)], DATASET_LABELS[str(d_)]) for m_, d_ in combined.index],
        names=['Model', 'Config']
    )
    return combined

auroc_comparison = build_metric_comparison('auroc')
auprc_comparison = build_metric_comparison('auprc')
loss07_comparison = build_metric_comparison('loss_07')
loss08_comparison = build_metric_comparison('loss_08')
loss09_comparison = build_metric_comparison('loss_09')

# --- Delta tables: Baseline / Lag-1 as absolute values, all other specs as delta vs baseline ---
# Negative delta = improvement over main results.
def build_metric_delta(metric):
    base = robust_metrics['11a_base_lag1'].set_index(['model', 'dataset'])[metric]
    frames = [base.round(3).rename(ROBUST_LABELS['11a_base_lag1'])]  # absolute baseline column
    for spec_key in ALL_SPEC_KEYS[1:]:  # skip 11a_base_lag1 — already added as absolute
        # A53: matched test years (pre-crisis specs share no test year with the baseline: NaN)
        delta = matched_delta(spec_key, metric).reindex(
            [(str(a), str(b)) for a, b in base.index]).set_axis(base.index).round(3).rename(ROBUST_LABELS[spec_key])
        frames.append(delta)
    combined = pd.concat(frames, axis=1)
    combined.index = pd.MultiIndex.from_tuples(
        [(MODEL_LABELS[str(m_)], DATASET_LABELS[str(d_)]) for m_, d_ in combined.index],
        names=['Model', 'Config']
    )
    return combined

loss07_delta = build_metric_delta('loss_07')
loss08_delta = build_metric_delta('loss_08')
loss09_delta = build_metric_delta('loss_09')

print("\n=== AUROC across robustness specs ===")
print(auroc_comparison.to_string())
print("\n=== AUPRC across robustness specs ===")
print(auprc_comparison.to_string())
print("\n=== L(0.8) across robustness specs ===")
print(loss08_comparison.to_string())


# --- DeLong summary + asterisk maps per spec ---
robust_delong = {}
for spec_key, (path, variant, n_yrs) in ROBUST_SPECS.items():
    summary = load_spec_delong(path, variant)
    summary['n_years'] = n_yrs
    robust_delong[spec_key] = summary
    summary['sig_marker'] = summary.apply(asterisk, axis=1)
    print(f"\n=== DeLong: {ROBUST_LABELS[spec_key]} (n_test_years={n_yrs}) ===")
    print(summary[['comparison', 'model_label', 'mean_z', 'pct_sig', 'n', 'sig_marker']].to_string(index=False))

    amap = summary.pivot(index='model_label', columns='comparison', values=['mean_z', 'pct_sig', 'sig_marker'])
    amap.index.name = 'Model'
    amap.columns.names = [None, None]
    print(f"\n--- Asterisk map: {ROBUST_LABELS[spec_key]} ---")
    print(amap.to_string())

In [ ]:
# Export all robustness tables to a dedicated Excel file
OUT_ROBUST = BASE / 'outputs/tables'

with pd.ExcelWriter(OUT_ROBUST / 'results_15_robust.xlsx', engine='openpyxl') as writer:
    auroc_comparison.to_excel(writer,  sheet_name='auroc_comparison')
    auprc_comparison.to_excel(writer,  sheet_name='auprc_comparison')
    loss07_comparison.to_excel(writer, sheet_name='loss07_comparison')
    loss08_comparison.to_excel(writer, sheet_name='loss08_comparison')
    loss09_comparison.to_excel(writer, sheet_name='loss09_comparison')
    loss07_delta.to_excel(writer, sheet_name='loss07_delta', float_format='%.3f')
    loss08_delta.to_excel(writer, sheet_name='loss08_delta', float_format='%.3f')
    loss09_delta.to_excel(writer, sheet_name='loss09_delta', float_format='%.3f')
    for spec_key, summary in robust_delong.items():
        # Shorten sheet name to fit Excel's 31-char limit
        short = spec_key.replace('_base_lag', '_bl').replace('_robust_lag', '_rl').replace('_fixed', '_f').replace('1_lag1', '1_l1')
        summary.to_excel(writer, sheet_name=f'delong_{short}', index=False)

print(f"Saved → {OUT_ROBUST / 'results_15_robust.xlsx'}")

## NB11b vs NB11a — Extended indicator set effect (Lag-1)

In [ ]:
# AUROC delta: NB11b (Extended / Lag-1) minus NB11a (Baseline / Lag-1)
# Shows the effect of expanding the indicator set while keeping the lag fixed at t-1.
# Positive = extended indicators improve AUROC over the baseline specification.
# Vertical layout (3|1): each config gets its own row, more horizontal space per panel.

base_auroc = robust_metrics['11a_base_lag1'].set_index(['model', 'dataset'])['auroc']
ext_auroc  = robust_metrics['11b_robust_lag1'].set_index(['model', 'dataset'])['auroc']

fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharey=False)

for ax, config, color in zip(axes,
                              ['micro', 'macro', 'integrated'],
                              [DATASET_COLORS['micro'], DATASET_COLORS['macro'], DATASET_COLORS['integrated']]):
    x = np.arange(len(MODEL_ORDER))
    vals = []
    for mdl in MODEL_ORDER:
        key = (mdl, config)
        if key in ext_auroc.index and key in base_auroc.index:
            vals.append(float(ext_auroc.loc[key]) - float(base_auroc.loc[key]))
        else:
            vals.append(np.nan)

    bars = ax.bar(x, vals, 0.6, color=color, edgecolor='white', linewidth=0.5, alpha=0.85)
    ax.axhline(0, color='black', lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[mdl] for mdl in MODEL_ORDER], rotation=25, ha='right', fontsize=9)
    ax.set_title(DATASET_LABELS[config], fontsize=10, fontweight='bold')
    ax.set_ylabel('ΔAUROC  (Extended − Baseline / Lag-1)', fontsize=9)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.margins(y=0.2)  # extra padding so labels near axis edges are not clipped

    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    v + (0.003 if v >= 0 else -0.003),
                    f'{v:+.3f}', ha='center',
                    va='bottom' if v >= 0 else 'top', fontsize=8)

fig.suptitle(
    'AUROC Change: Extended vs Baseline Indicator Set at Lag-1\n'
    'Positive = improvement over baseline specification',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUTPUT / 'nb11b_vs_nb11a_auroc_delta.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → nb11b_vs_nb11a_auroc_delta.png")

In [ ]:
# AUPRC delta: NB11b (Extended / Lag-1) minus NB11a (Baseline / Lag-1)
# Shows the effect of expanding the indicator set while keeping the lag fixed at t-1.
# Positive = extended indicators improve AUPRC over the baseline specification.
# Vertical layout (3x1): each config gets its own row, more horizontal space per panel.

base_auprc = robust_metrics['11a_base_lag1'].set_index(['model', 'dataset'])['auprc']
ext_auprc  = robust_metrics['11b_robust_lag1'].set_index(['model', 'dataset'])['auprc']

fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharey=False)

for ax, config, color in zip(axes,
                              ['micro', 'macro', 'integrated'],
                              [DATASET_COLORS['micro'], DATASET_COLORS['macro'], DATASET_COLORS['integrated']]):
    x = np.arange(len(MODEL_ORDER))
    vals = []
    for mdl in MODEL_ORDER:
        key = (mdl, config)
        if key in ext_auprc.index and key in base_auprc.index:
            vals.append(float(ext_auprc.loc[key]) - float(base_auprc.loc[key]))
        else:
            vals.append(np.nan)

    bars = ax.bar(x, vals, 0.6, color=color, edgecolor='white', linewidth=0.5, alpha=0.85)
    ax.axhline(0, color='black', lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[mdl] for mdl in MODEL_ORDER], rotation=25, ha='right', fontsize=9)
    ax.set_title(DATASET_LABELS[config], fontsize=10, fontweight='bold')
    ax.set_ylabel('DAUPRC  (Extended - Baseline / Lag-1)', fontsize=9)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.margins(y=0.2)  # extra padding so labels near axis edges are not clipped

    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    v + (0.003 if v >= 0 else -0.003),
                    f'{v:+.3f}', ha='center',
                    va='bottom' if v >= 0 else 'top', fontsize=8)

fig.suptitle(
    'AUPRC Change: Extended vs Baseline Indicator Set at Lag-1'
    'Positive = improvement over baseline specification',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUTPUT / 'nb11b_vs_nb11a_auprc_delta.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → nb11b_vs_nb11a_auprc_delta.png")

In [ ]:
# Combined AUROC + AUPRC delta: NB11b (Extended / Lag-1) minus NB11a (Baseline / Lag-1)
# Grouped bars per model: AUROC delta (full alpha) and AUPRC delta (lighter shade, same config colour).
# Vertical layout (3x1): one panel per config.

import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple

base_auroc = robust_metrics['11a_base_lag1'].set_index(['model', 'dataset'])['auroc']
ext_auroc  = robust_metrics['11b_robust_lag1'].set_index(['model', 'dataset'])['auroc']
base_auprc = robust_metrics['11a_base_lag1'].set_index(['model', 'dataset'])['auprc']
ext_auprc  = robust_metrics['11b_robust_lag1'].set_index(['model', 'dataset'])['auprc']

fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharey=False)
width = 0.35

for ax, config in zip(axes, ['micro', 'macro', 'integrated']):
    color = DATASET_COLORS[config]
    x = np.arange(len(MODEL_ORDER))

    vals_auroc, vals_auprc = [], []
    for mdl in MODEL_ORDER:
        key = (mdl, config)
        vals_auroc.append(
            float(ext_auroc.loc[key]) - float(base_auroc.loc[key])
            if key in ext_auroc.index and key in base_auroc.index else np.nan
        )
        vals_auprc.append(
            float(ext_auprc.loc[key]) - float(base_auprc.loc[key])
            if key in ext_auprc.index and key in base_auprc.index else np.nan
        )

    bars1 = ax.bar(x - width/2, vals_auroc, width, color=color, alpha=0.85,
                   edgecolor='white', linewidth=0.5)
    bars2 = ax.bar(x + width/2, vals_auprc,  width, color=color, alpha=0.4,
                   edgecolor='white', linewidth=0.5)

    ax.axhline(0, color='black', lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[mdl] for mdl in MODEL_ORDER], rotation=25, ha='right', fontsize=9)
    ax.set_title(DATASET_LABELS[config], fontsize=10, fontweight='bold')
    ax.set_ylabel('Delta  (Extended - Baseline / Lag-1)', fontsize=9)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.margins(y=0.2)

    for bar, v in zip([*bars1, *bars2], [*vals_auroc, *vals_auprc]):
        if not np.isnan(v) and abs(v) >= 0.005:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    v + (0.003 if v >= 0 else -0.003),
                    f'{v:+.3f}', ha='center',
                    va='bottom' if v >= 0 else 'top', fontsize=7.5)

patches_auroc = tuple(mpatches.Patch(color=DATASET_COLORS[c], alpha=0.85) for c in ['micro', 'macro', 'integrated'])
patches_auprc = tuple(mpatches.Patch(color=DATASET_COLORS[c], alpha=0.4)  for c in ['micro', 'macro', 'integrated'])

fig.legend(
    handles=[patches_auroc, patches_auprc],
    labels=['AUROC delta', 'AUPRC delta'],
    handler_map={tuple: HandlerTuple(ndivide=None, pad=0.1)},
    fontsize=9, loc='lower center', ncol=2, bbox_to_anchor=(0.5, 0.01)
)

#fig.suptitle(
#    'AUROC and AUPRC Change: Extended vs Baseline Indicator Set at Lag-1\n',
#    fontsize=11, fontweight='bold'
#)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(OUTPUT / 'nb11b_vs_nb11a_combined_delta.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → nb11b_vs_nb11a_combined_delta.png")

## Lag effect — NB11c vs NB11a and NB11d vs NB11b

In [ ]:
# Lag effect: what does moving from t-1 to t-2 do, within each indicator set?
# Two paired comparisons — indicator set held constant, only lag changes:
#   NB11c − NB11a: baseline indicators, lag-2 vs lag-1  (full color)
#   NB11d − NB11b: extended indicators, lag-2 vs lag-1  (lighter shade, same color)
#
# Note: NB11a/b are averaged over 3 test years (2010–2012); NB11c/d over 2 (2011–2012).
# The delta therefore conflates a small year-set difference — treated as directional only.
#
# sharey=False: micro values are small relative to macro/integrated — each panel auto-scales.
# Vertical layout (3|1): each config gets its own row, more horizontal space per panel.

import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple

base_lag1 = robust_metrics['11a_base_lag1'].set_index(['model', 'dataset'])['auroc']
base_lag2 = robust_metrics['11c_base_lag2'].set_index(['model', 'dataset'])['auroc']
ext_lag1  = robust_metrics['11b_robust_lag1'].set_index(['model', 'dataset'])['auroc']
ext_lag2  = robust_metrics['11d_robust_lag2'].set_index(['model', 'dataset'])['auroc']

fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharey=False)
width = 0.35

for ax, config in zip(axes, ['micro', 'macro', 'integrated']):
    color = DATASET_COLORS[config]
    x = np.arange(len(MODEL_ORDER))

    vals_base, vals_ext = [], []
    for mdl in MODEL_ORDER:
        key = (mdl, config)
        vals_base.append(
            float(base_lag2.loc[key]) - float(base_lag1.loc[key])
            if key in base_lag2.index and key in base_lag1.index else np.nan
        )
        vals_ext.append(
            float(ext_lag2.loc[key]) - float(ext_lag1.loc[key])
            if key in ext_lag2.index and key in ext_lag1.index else np.nan
        )

    bars1 = ax.bar(x - width/2, vals_base, width, color=color, alpha=0.85,
                   edgecolor='white', linewidth=0.5)
    bars2 = ax.bar(x + width/2, vals_ext,  width, color=color, alpha=0.4,
                   edgecolor='white', linewidth=0.5)

    ax.axhline(0, color='black', lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[mdl] for mdl in MODEL_ORDER], rotation=25, ha='right', fontsize=9)
    ax.set_title(DATASET_LABELS[config], fontsize=10, fontweight='bold')
    ax.set_ylabel('ΔAUROC  (Lag-2 − Lag-1)', fontsize=9)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

    # 2 decimal places; skip labels where |delta| < 0.01
    for bar, v in zip([*bars1, *bars2], [*vals_base, *vals_ext]):
        if not np.isnan(v) and abs(v) >= 0.01:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    v + (0.003 if v >= 0 else -0.003),
                    f'{v:+.2f}', ha='center',
                    va='bottom' if v >= 0 else 'top', fontsize=8)

# Split-color legend: each swatch shows blue/orange/green at full and light alpha
patches_base = tuple(mpatches.Patch(color=DATASET_COLORS[c], alpha=0.85) for c in ['micro', 'macro', 'integrated'])
patches_ext  = tuple(mpatches.Patch(color=DATASET_COLORS[c], alpha=0.4)  for c in ['micro', 'macro', 'integrated'])

fig.legend(
    handles=[patches_base, patches_ext],
    labels=['Baseline indicators  (lag-2 − lag-1 specification)', 'Extended indicators  (lag-2 specification  − lag-1 specification )'],
    handler_map={tuple: HandlerTuple(ndivide=None, pad=0.1)},
    fontsize=9, loc='lower center', ncol=2, bbox_to_anchor=(0.5, 0.01)
)

#fig.suptitle(
#    'AUROC Change: Lag-2 vs Lag-1 within Each Indicator Set\n'
#    'Positive = improvement at longer prediction horizon',
#    fontsize=11, fontweight='bold'
#)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(OUTPUT / 'lag_effect_auroc_delta.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → lag_effect_auroc_delta.png")

## RT1 delta — onset year exclusion effect

In [ ]:
# AUROC delta when crisis onset year is excluded from training (RT1 vs baseline_t1).
# Baseline restricted to 2011-2012 so both variants use identical test years.
# Delta pooled across models; directional interpretation only (n=2 windows, no significance test).

RT1_SPEC_KEYS = [
    ('11a_base_lag1',   'Baseline / Lag-1'),
    ('11b_robust_lag1', 'Baseline Extended / Lag-1'),
    ('11c_base_lag2',   'Baseline / Lag-2'),
    ('11d_robust_lag2', 'Baseline Extended / Lag-2'),
]
OVERLAP_YEARS = [2011, 2012]  # years present in both baseline_t1 and rt1 across all specs

rt1_records = []
for spec_key, spec_label in RT1_SPEC_KEYS:
    df = pd.read_excel(RESULT_FILES[spec_key], sheet_name='all_results')
    df = df[
        df['variant'].isin(['baseline_t1', 'rt1']) &
        df['test_year'].isin(OVERLAP_YEARS) &
        (df['model'] != 'lr_no_dummies')
    ]
    # Mean AUROC per model x dataset x variant across the two overlapping years
    agg = df.groupby(['model', 'dataset', 'variant'])['auroc'].mean().reset_index()
    pivot = agg.pivot_table(index=['model', 'dataset'], columns='variant', values='auroc').reset_index()
    pivot['delta'] = pivot['rt1'] - pivot['baseline_t1']
    pivot['spec'] = spec_label
    rt1_records.append(pivot)

rt1_df = pd.concat(rt1_records, ignore_index=True)

# Pool across models: mean delta per dataset x spec
pooled = rt1_df.groupby(['spec', 'dataset'])['delta'].mean().reset_index()

CONFIG_COLOURS = {'micro': '#1f77b4', 'macro': '#ff7f0e', 'integrated': '#2ca02c'}
CONFIG_LABELS  = {'micro': 'Micro-only', 'macro': 'Macro-only', 'integrated': 'Integrated'}
spec_order     = [label for _, label in RT1_SPEC_KEYS]
dataset_order  = ['micro', 'macro', 'integrated']
x, width       = np.arange(len(spec_order)), 0.22

fig, ax = plt.subplots(figsize=(10, 5))
for i, dataset in enumerate(dataset_order):
    sub  = pooled[pooled['dataset'] == dataset].set_index('spec')
    vals = [sub.loc[sp, 'delta'] if sp in sub.index else 0.0 for sp in spec_order]
    off  = (i - (len(dataset_order) - 1) / 2) * width
    bars = ax.bar(x + off, vals, width, label=CONFIG_LABELS[dataset],
                  color=CONFIG_COLOURS[dataset], edgecolor='white', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                v + (0.002 if v >= 0 else -0.002),
                f'{v:+.3f}', ha='center',
                va='bottom' if v >= 0 else 'top', fontsize=7.5)

ax.axhline(0, color='black', linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(spec_order, fontsize=10)
ax.set_ylabel('Mean ΔAUROC  (Onset excluded − baseline)', fontsize=10)
#ax.set_title(
    #'AUROC change when crisis onset & t+1 years are excluded from training',
    #'Pooled across models  |  baseline restricted to 2011–2012  |  directional only, n = 2 test windows',
#    fontsize=10
#)
ax.legend(fontsize=9)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(OUTPUT / 'rt1_delta_auroc.png', dpi=150, bbox_inches='tight')
plt.show()
print(pooled.pivot(index='spec', columns='dataset', values='delta').round(3).to_string())

AUPRC version

In [ ]:
# AUPRC delta when crisis onset year is excluded from training (RT1 vs baseline_t1).
# Baseline restricted to 2011-2012 so both variants use identical test years.
# Delta pooled across models; directional interpretation only (n=2 windows, no significance test).

RT1_SPEC_KEYS = [
    ('11a_base_lag1',   'Baseline / Lag-1'),
    ('11b_robust_lag1', 'Baseline Extended / Lag-1'),
    ('11c_base_lag2',   'Baseline / Lag-2'),
    ('11d_robust_lag2', 'Baseline Extended / Lag-2'),
]
OVERLAP_YEARS = [2011, 2012]  # years present in both baseline_t1 and rt1 across all specs

rt1_records = []
for spec_key, spec_label in RT1_SPEC_KEYS:
    df = pd.read_excel(RESULT_FILES[spec_key], sheet_name='all_results')
    df = df[
        df['variant'].isin(['baseline_t1', 'rt1']) &
        df['test_year'].isin(OVERLAP_YEARS) &
        (df['model'] != 'lr_no_dummies')
    ]
    # Mean AUPRC per model x dataset x variant across the two overlapping years
    agg = df.groupby(['model', 'dataset', 'variant'])['auprc'].mean().reset_index()
    pivot = agg.pivot_table(index=['model', 'dataset'], columns='variant', values='auprc').reset_index()
    pivot['delta'] = pivot['rt1'] - pivot['baseline_t1']
    pivot['spec'] = spec_label
    rt1_records.append(pivot)

rt1_df = pd.concat(rt1_records, ignore_index=True)

# Pool across models: mean delta per dataset x spec
pooled = rt1_df.groupby(['spec', 'dataset'])['delta'].mean().reset_index()

CONFIG_COLOURS = {'micro': '#1f77b4', 'macro': '#ff7f0e', 'integrated': '#2ca02c'}
CONFIG_LABELS  = {'micro': 'Micro-only', 'macro': 'Macro-only', 'integrated': 'Integrated'}
spec_order     = [label for _, label in RT1_SPEC_KEYS]
dataset_order  = ['micro', 'macro', 'integrated']
x, width       = np.arange(len(spec_order)), 0.22

fig, ax = plt.subplots(figsize=(10, 5))
for i, dataset in enumerate(dataset_order):
    sub  = pooled[pooled['dataset'] == dataset].set_index('spec')
    vals = [sub.loc[sp, 'delta'] if sp in sub.index else 0.0 for sp in spec_order]
    off  = (i - (len(dataset_order) - 1) / 2) * width
    bars = ax.bar(x + off, vals, width, label=CONFIG_LABELS[dataset],
                  color=CONFIG_COLOURS[dataset], edgecolor='white', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                v + (0.002 if v >= 0 else -0.002),
                f'{v:+.3f}', ha='center',
                va='bottom' if v >= 0 else 'top', fontsize=7.5)

ax.axhline(0, color='black', linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(spec_order, fontsize=10)
ax.set_ylabel('Mean ΔAUPRC  (Onset excluded − baseline)', fontsize=10)
#    ax.set_title(
#        'AUPRC change when crisis onset & t+1 years are excluded from training',
#        #'Pooled across models  |  baseline restricted to 2011–2012  |  directional only, n = 2 test windows',
#        fontsize=10
#    )
ax.legend(fontsize=9)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(OUTPUT / 'rt1_delta_auprc.png', dpi=150, bbox_inches='tight')
plt.show()
print(pooled.pivot(index='spec', columns='dataset', values='delta').round(3).to_string())

## NB12a/b (Macro-only)

In [ ]:
# NB12a/b: macro-only models run over separate windows — go to appendix in thesis
# 12a uses baseline window (variant: baseline_t1), 12b uses robust window
# No DeLong test here — NB12 is macro-only so the integrated vs micro/macro comparison
# does not apply; DeLong is only meaningful for the 3-config specs (NB11a/b/c/d, RT2)
NB12_SPECS = {
    '12a_macro_base':   (RESULT_FILES['12a_macro_base'],  'macro_standalone'),
    '12b_macro_robust': (RESULT_FILES['12b_macro_robust'], 'macro_standalone'),
}
NB12_LABELS = {
    '12a_macro_base':   'Baseline',
    '12b_macro_robust': 'Extended',
}

nb12_metrics = {}
for spec_key, (path, variant) in NB12_SPECS.items():
    nb12_metrics[spec_key] = load_spec_metrics(path, variant)
    print(f"Loaded {spec_key}: {len(nb12_metrics[spec_key])} rows")

# --- Combined AUROC + AUPRC table ---
# Columns: AUROC (Baseline), AUPRC (Baseline), AUROC (Extended), AUPRC (Extended)
perf_frames = []
for metric in ['auroc', 'auprc']:
    for spec_key in NB12_SPECS:
        col_label = f"{metric.upper()} ({NB12_LABELS[spec_key]})"
        df = nb12_metrics[spec_key][['model', 'dataset', metric]].rename(columns={metric: col_label})
        perf_frames.append(df.set_index(['model', 'dataset']))

nb12_perf = pd.concat(perf_frames, axis=1)
# Reorder: group by spec (Baseline cols first, then Extended)
col_order = (
    [f"AUROC ({NB12_LABELS['12a_macro_base']})", f"AUPRC ({NB12_LABELS['12a_macro_base']})"] +
    [f"AUROC ({NB12_LABELS['12b_macro_robust']})", f"AUPRC ({NB12_LABELS['12b_macro_robust']})"]
)
nb12_perf = nb12_perf[col_order]
nb12_perf.index = pd.MultiIndex.from_tuples(
    [(MODEL_LABELS[str(m_)], DATASET_LABELS[str(d_)]) for m_, d_ in nb12_perf.index],
    names=['Model', 'Config']
)

print("\n=== NB12 AUROC + AUPRC ===")
print(nb12_perf.to_string())

# --- Combined loss table ---
# Columns grouped by spec: L(0.7)/L(0.8)/L(0.9) for Baseline, then same for Extended
loss_frames = []
for spec_key in NB12_SPECS:
    for loss_col, label in [('loss_07', 'L(0.7)'), ('loss_08', 'L(0.8)'), ('loss_09', 'L(0.9)')]:
        col_label = f"{label} ({NB12_LABELS[spec_key]})"
        df = nb12_metrics[spec_key][['model', 'dataset', loss_col]].rename(columns={loss_col: col_label})
        loss_frames.append(df.set_index(['model', 'dataset']))

nb12_loss_combined = pd.concat(loss_frames, axis=1)
nb12_loss_combined.index = pd.MultiIndex.from_tuples(
    [(MODEL_LABELS[str(m_)], DATASET_LABELS[str(d_)]) for m_, d_ in nb12_loss_combined.index],
    names=['Model', 'Config']
)

print("\n=== NB12 Loss functions ===")
print(nb12_loss_combined.to_string())

# Export: two combined sheets replace the previous five separate ones
with pd.ExcelWriter(OUT_ROBUST / 'results_15_robust.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    nb12_perf.to_excel(writer, sheet_name='nb12_auroc_auprc')
    nb12_loss_combined.to_excel(writer, sheet_name='nb12_loss')

print(f"\nSaved NB12 sheets → {OUT_ROBUST / 'results_15_robust.xlsx'}")

## training window effect

In [ ]:
# Timeframe effect: NB12 (panels from 1999 baseline / 1995 extended) vs NB11 (2004–2017), macro-only features.
# A55: earlier titles said 2001–2017, which matches neither panel. The comparison also differs in
# granularity (country-year vs bank-year), so it is not a pure window effect.
# Isolates the training window contribution by holding the feature set constant.

nb11a_macro = m[m['dataset'] == 'macro'][['model', 'auroc', 'auprc']].copy()
nb11b_macro = robust_metrics['11b_robust_lag1']
nb11b_macro = nb11b_macro[nb11b_macro['dataset'] == 'macro'][['model', 'auroc', 'auprc']].copy()
# A44: NB12 tests 2008-2012, NB11 2010-2012; compare on the shared test years only
SHARED_TEST_YEARS = [2010, 2011, 2012]
nb12a_macro = load_spec_metrics(RESULT_FILES['12a_macro_base'], 'macro_standalone', years=SHARED_TEST_YEARS)[['model', 'auroc', 'auprc']].copy()
nb12b_macro = load_spec_metrics(RESULT_FILES['12b_macro_robust'], 'macro_standalone', years=SHARED_TEST_YEARS)[['model', 'auroc', 'auprc']].copy()

# Categorical → str so .loc indexing works cleanly
for df in [nb11a_macro, nb11b_macro, nb12a_macro, nb12b_macro]:
    df['model'] = df['model'].astype(str)

x     = np.arange(len(MODEL_ORDER))
width = 0.2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, ylabel in zip(axes, ['auroc', 'auprc'], ['AUROC', 'AUPRC']):
    def vals(df):
        return [df.set_index('model').loc[mdl, metric] for mdl in MODEL_ORDER]

    specs = [
        (vals(nb11a_macro), '#4C72B0', '2004–2017 (Baseline)'),
        (vals(nb12a_macro), '#87AECE', 'Macro standalone — 1999–2017 (Baseline)'),
        (vals(nb11b_macro), '#DD8452', '2004–2017 (Extended)'),
        (vals(nb12b_macro), '#EFB88A', 'Macro standalone — 1995–2017 (Extended)'),
    ]
    for (v, color, label), offset in zip(specs, [-1.5, -0.5, 0.5, 1.5]):
        ax.bar(x + offset * width, v, width, label=label, color=color,
               edgecolor='white', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[mdl] for mdl in MODEL_ORDER], rotation=30,
                       ha='right', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(f'{ylabel} — Macro-only, Timeframe Effect', fontsize=10, fontweight='bold')
    ax.set_ylim(0.4, 1.0)
    ax.axhline(0.5, color='gray', lw=0.8, linestyle='--', alpha=0.5)
    ax.legend(fontsize=8, loc='lower right')

fig.suptitle(
    'Extended Window Effect: Macro standalone (from 1999 / 1995) vs baseline specifications (2004–2017)\n'
    'Same macro feature set, test years 2010–2012; the comparison also differs in granularity',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUTPUT / 'nb12_vs_nb11_timeframe_effect.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → nb12_vs_nb11_timeframe_effect.png")

In [ ]:
# AUROC gain from extended window: NB12 minus NB11, macro-only features

delta_base   = [nb12a_macro.set_index('model').loc[mdl, 'auroc'] -
nb11a_macro.set_index('model').loc[mdl, 'auroc'] for mdl in MODEL_ORDER]
delta_robust = [nb12b_macro.set_index('model').loc[mdl, 'auroc'] -
nb11b_macro.set_index('model').loc[mdl, 'auroc'] for mdl in MODEL_ORDER]

x, width = np.arange(len(MODEL_ORDER)), 0.35
fig, ax  = plt.subplots(figsize=(10, 5))

bars1 = ax.bar(x - width/2, delta_base,   width, label='Macro standalone − Baseline specification (t-1)',
color='#4C72B0', edgecolor='white')
bars2 = ax.bar(x + width/2, delta_robust, width, label='Macro standalone extended − Baseline extended specification (t-1)',
color='#DD8452', edgecolor='white')

ax.axhline(0, color='black', lw=0.9)
ax.set_xticks(x)
ax.set_xticklabels([MODEL_LABELS[mdl] for mdl in MODEL_ORDER], rotation=30,
ha='right', fontsize=9)
ax.set_ylabel('AUROC gain  (extended − standard window)', fontsize=10)
ax.set_title('Training Window Effect — Macro-only AUROC\nMacro standalone (from 1999 / 1995) minus baseline specifications (t-1; 2004–2017), test years 2010–2012', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

for bar in [*bars1, *bars2]:
    h   = bar.get_height()
    off = 0.003 if h >= 0 else -0.003
    ax.text(bar.get_x() + bar.get_width() / 2, h + off, f'{h:+.3f}',
            ha='center', va='bottom' if h >= 0 else 'top', fontsize=7.5)

plt.tight_layout()
plt.savefig(OUTPUT / 'nb12_vs_nb11_delta.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → nb12_vs_nb11_delta.png")

In [ ]:
# FIGURE 5 (GRANULARITY-ONLY): country-level macro-only (NB12 isolation, 2005-2017)
# minus bank-level macro config (NB11), on the SAME window (test 2010-2012), so the ONLY
# difference is GRANULARITY (country-year vs bank-year + LR country dummies). This replaces
# the earlier confounded extended-WINDOW comparison (2001-2017), which mixed window +
# granularity + dummies. Positive delta = country-level BETTER than bank-level.
# NB: the LR delta reflects the country-dummy artifact (bank-level LR carries country dummies
# the country-level model cannot), NOT granularity per se -- caveat this in the text.
# Reads the isolation runs (12a/12b_macro_..._isolate_gran); the bank sides (nb11a/b_macro)
# are reused from the cell above.

import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple

# Load granularity-isolation macro-only results (variant 'macro_standalone', macro dataset)
_iso_a = load_spec_metrics(
    BASE / 'outputs/results/12a_gran/results_12a_isolate_gran_final.xlsx',
    'macro_standalone')
_iso_b = load_spec_metrics(
    BASE / 'outputs/results/12b_gran/results_12b_isolate_gran_final.xlsx',
    'macro_standalone')
nb12a_iso = _iso_a[_iso_a['dataset'] == 'macro'][['model', 'auroc', 'auprc']].copy()
nb12b_iso = _iso_b[_iso_b['dataset'] == 'macro'][['model', 'auroc', 'auprc']].copy()
for _df in [nb12a_iso, nb12b_iso]:
    _df['model'] = _df['model'].astype(str)

color = DATASET_COLORS['macro']

fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharey=False)

comparisons = [
    (axes[0], nb12a_iso, nb11a_macro,
     'Baseline set'),
    (axes[1], nb12b_iso, nb11b_macro,
     'Extended set '),
]

x = np.arange(len(MODEL_ORDER))
width = 0.35

for ax, df_nb12, df_nb11, title in comparisons:
    nb12_idx = df_nb12.set_index('model')
    nb11_idx = df_nb11.set_index('model')

    vals_auroc = [nb12_idx.loc[mdl, 'auroc'] - nb11_idx.loc[mdl, 'auroc'] for mdl in MODEL_ORDER]
    vals_auprc = [nb12_idx.loc[mdl, 'auprc'] - nb11_idx.loc[mdl, 'auprc'] for mdl in MODEL_ORDER]

    bars1 = ax.bar(x - width/2, vals_auroc, width, color=color, alpha=0.85,
                   edgecolor='white', linewidth=0.5)
    bars2 = ax.bar(x + width/2, vals_auprc, width, color=color, alpha=0.4,
                   edgecolor='white', linewidth=0.5)

    ax.axhline(0, color='black', lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[mdl] for mdl in MODEL_ORDER], rotation=25, ha='right', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel('Delta (Country-level − Bank-level)', fontsize=9)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.margins(y=0.2)

    for bar, v in zip([*bars1, *bars2], [*vals_auroc, *vals_auprc]):
        if not np.isnan(v) and abs(v) >= 0.005:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    v + (0.003 if v >= 0 else -0.003),
                    f'{v:+.3f}', ha='center',
                    va='bottom' if v >= 0 else 'top', fontsize=8)

patch_auroc = tuple(mpatches.Patch(color=color, alpha=0.85) for _ in ['macro'])
patch_auprc = tuple(mpatches.Patch(color=color, alpha=0.4)  for _ in ['macro'])

fig.legend(
    handles=[patch_auroc, patch_auprc],
    labels=['AUROC delta', 'AUPRC delta'],
    handler_map={tuple: HandlerTuple(ndivide=None, pad=0.1)},
    fontsize=9, loc='lower center', ncol=2, bbox_to_anchor=(0.5, 0.01)
)

#fig.suptitle(
#    'Granularity comparison: country-level vs bank-level macro-only (same 2005–2017 window)\n'
#    'AUROC and AUPRC change',
#    fontsize=11, fontweight='bold'
#)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(OUTPUT / 'nb12_vs_nb11_combined_delta.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → nb12_vs_nb11_combined_delta.png (GRANULARITY-only version)")

In [ ]:
# A42: like-for-like granularity. Figure 4 scores the country-level model on 25 countries and the
# bank-level model on several hundred bank rows, where a country weighs by its number of banks. Here both
# are scored on the same unit: bank-level macro predictions are averaged per country-year and AUROC is
# computed over the same country-years as the country-level model. Reported with and without LR, because
# only the bank-level LR carries country dummies (the reason the thesis leaves LR out of this reading).
# The test sets are df[year == test_year] without row drops, so each panel's country column lines up
# with the probability lists in its proba_store.
from sklearn.metrics import roc_auc_score as _auroc

LFL_SPECS = {
    'Baseline set': (PROBA_FILES['11a_base_lag1'],
                     BASE / 'data/engineered/parquet_saves/me_merged_base.parquet',
                     BASE / 'outputs/results/12a_gran/proba_store_final.pkl',
                     BASE / 'data/individual_merges/parquet_saves/macro_ind_base_me.parquet'),
    'Extended set': (PROBA_FILES['11b_robust_lag1'],
                     BASE / 'data/engineered/parquet_saves/me_merged_robust.parquet',
                     BASE / 'outputs/results/12b_gran/proba_store_final.pkl',
                     BASE / 'data/individual_merges/parquet_saves/macro_ind_robust_me.parquet'),
}
LFL_YEARS = [2010, 2011, 2012]

lfl_rows = []
for set_name, (bank_ps_path, bank_panel, cty_ps_path, cty_panel) in LFL_SPECS.items():
    with open(bank_ps_path, 'rb') as fh:
        bank_ps = pickle.load(fh)
    with open(cty_ps_path, 'rb') as fh:
        cty_ps = pickle.load(fh)
    bank_df = pd.read_parquet(bank_panel, columns=['country_iso', 'year'])
    cty_df  = pd.read_parquet(cty_panel,  columns=['country_iso', 'year'])
    for mdl in MODEL_ORDER:
        for yr in LFL_YEARS:
            kb, kc = (mdl, 'macro', yr, 'baseline_t1'), (mdl, 'macro', yr, 'macro_standalone')
            if kb not in bank_ps or kc not in cty_ps:
                continue
            yb, pb = bank_ps[kb]
            yc, pc = cty_ps[kc]
            cb = bank_df.loc[bank_df['year'] == yr, 'country_iso'].to_numpy()
            cc = cty_df.loc[cty_df['year'] == yr, 'country_iso'].to_numpy()
            assert len(cb) == len(yb) and len(cc) == len(yc), f'row mismatch {set_name} {mdl} {yr}'
            bank_cy = (pd.DataFrame({'country_iso': cb, 'y': yb, 'p': pb})
                         .groupby('country_iso').agg(y=('y', 'max'), p_bank=('p', 'mean')))
            both = bank_cy.join(pd.DataFrame({'country_iso': cc, 'p_country': pc}).set_index('country_iso'), how='inner')
            if both['y'].nunique() < 2:
                continue
            lfl_rows.append({
                'set': set_name, 'model': mdl, 'test_year': yr, 'n_countries': len(both),
                'bank_rows':        _auroc(yb, pb),                      # Figure 4's bank side
                'bank_per_country': _auroc(both['y'], both['p_bank']),   # same unit as the country model
                'country_model':    _auroc(both['y'], both['p_country']),
            })

lfl = pd.DataFrame(lfl_rows)
lfl_model = lfl.groupby(['set', 'model'])[['bank_rows', 'bank_per_country', 'country_model']].mean().round(3)
lfl_summary = pd.concat({
    'all six models': lfl_model.groupby('set').mean(),
    'without LR':     lfl_model.drop(index='logistic_regression', level='model').groupby('set').mean(),
}, names=['models']).round(3)
lfl_summary['country_minus_bank_per_country'] = (lfl_summary['country_model'] - lfl_summary['bank_per_country']).round(3)
print('=== A42: like-for-like granularity (mean AUROC over 2010-2012) ===')
print(lfl_summary.to_string())

with pd.ExcelWriter(OUT_ROBUST / 'granularity_like_for_like.xlsx', engine='openpyxl') as writer:
    lfl_summary.to_excel(writer, sheet_name='summary')
    lfl_model.to_excel(writer, sheet_name='per_model')
    lfl.to_excel(writer, sheet_name='per_year', index=False)
print(f"Saved → {OUT_ROBUST / 'granularity_like_for_like.xlsx'}")

In [ ]:

# Slope chart: one line per model showing NB11 → NB12 AUROC shift, macro-only

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
colors    = plt.rcParams['axes.prop_cycle'].by_key()['color']

panels = [
    (axes[0], nb11a_macro, nb12a_macro, 'Baseline specification → Macro standalone'),
    (axes[1], nb11b_macro, nb12b_macro, 'Baseline extended specification  → Macro standalone'),
]

for ax, df_l, df_r, title in panels:
    for i, mdl in enumerate(MODEL_ORDER):
        yl = df_l.set_index('model').loc[mdl, 'auroc']
        yr = df_r.set_index('model').loc[mdl, 'auroc']
        c  = colors[i % len(colors)]
        ax.plot([0, 1], [yl, yr], 'o-', color=c, lw=2, ms=6, label=MODEL_LABELS[mdl])
        ax.text(-0.06, yl, f'{yl:.3f}', ha='right', va='center', fontsize=8, color=c)
        ax.text( 1.06, yr, f'{yr:.3f}', ha='left',  va='center', fontsize=8, color=c)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Baseline\n(2004–2017)', 'Macro standalone\n(from 1999 / 1995)'], fontsize=10)
    ax.set_xlim(-0.35, 1.35)
    ax.set_ylabel('AUROC', fontsize=10)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

axes[1].legend(fontsize=8, loc='lower right')
fig.suptitle('Training Window Effect — Macro-only AUROC per Model\nSame feature set; macro standalone adds the years before 2004 (from 1999 baseline, 1995 extended)',
            fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT / 'nb12_vs_nb11_slope.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → nb12_vs_nb11_slope.png")

## Loss L(0.8 delta)

In [ ]:
# Loss L(0.8) delta: each robustness spec minus NB11a baseline
# Negative = improvement over main results. L(0.8) is representative — L(0.7) and L(0.9) follow same direction.

base_loss = robust_metrics['11a_base_lag1'].set_index(['model', 'dataset'])['loss_08']

DELTA_SPECS   = [k for k in ALL_SPEC_KEYS if k != '11a_base_lag1']
spec_colors   = plt.rcParams['axes.prop_cycle'].by_key()['color'][:len(DELTA_SPECS)]
n_specs       = len(DELTA_SPECS)
width         = 0.7 / n_specs

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, config in zip(axes, ['micro', 'macro', 'integrated']):
    x = np.arange(len(MODEL_ORDER))
    for i, (spec_key, color) in enumerate(zip(DELTA_SPECS, spec_colors)):
        spec_delta = matched_delta(spec_key, 'loss_08')  # A53: matched test years
        vals = [spec_delta.get((mdl, config), np.nan) for mdl in MODEL_ORDER]
        offset = (i - n_specs / 2 + 0.5) * width
        ax.bar(x + offset, vals, width, label=ROBUST_LABELS[spec_key],
                color=color, edgecolor='white', linewidth=0.4)

    ax.axhline(0, color='black', lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER], rotation=35, ha='right', fontsize=8)
    ax.set_title(DATASET_LABELS[config], fontsize=10, fontweight='bold')
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

axes[0].set_ylabel('L(0.8) delta vs baseline specification  (negative = improvement)', fontsize=9)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, fontsize=8, loc='lower center', ncol=n_specs,
            bbox_to_anchor=(0.5, -0.08))
fig.suptitle('L(0.8) Gain over Baseline specification — Extended Specs\nNegative = improvement',
            fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT / 'loss_delta_vs_nb11a.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → loss_delta_vs_nb11a.png")

In [ ]:
# Mean L(0.8) delta per spec per config — collapses model dimension to a mean bar.
# Primary axis is information type (config), matching the thesis focus on what data is fed
# rather than which model class performs best.

fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)
x = np.arange(len(DELTA_SPECS))
bar_colors = plt.rcParams['axes.prop_cycle'].by_key()['color'][:len(DELTA_SPECS)]

for ax, config in zip(axes, ['micro', 'macro', 'integrated']):
    means = []
    for spec_key in DELTA_SPECS:
        spec_delta = matched_delta(spec_key, 'loss_08')  # A53: matched test years
        spec_deltas = [spec_delta.get((mdl, config), np.nan) for mdl in MODEL_ORDER]
        means.append(np.nanmean(spec_deltas))

    ax.bar(x, means, 0.55, color=bar_colors, edgecolor='white', linewidth=0.5, alpha=0.85)
    ax.axhline(0, color='black', lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([ROBUST_LABELS[k] for k in DELTA_SPECS], rotation=35, ha='right', fontsize=8)
    ax.set_title(DATASET_LABELS[config], fontsize=10, fontweight='bold')
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

axes[0].set_ylabel('L(0.8) delta vs Baseline  (negative = improvement)', fontsize=9)
#fig.suptitle(
#    'Mean L(0.8) Change from Baseline specification \nMean across models; negative = improvement',
#    fontsize=11, fontweight='bold'
#)
plt.tight_layout()
plt.savefig(OUTPUT / 'loss_delta_vs_nb11a_v2.png', bbox_inches='tight', dpi=200)
plt.show()
print("Saved → loss_delta_vs_nb11a_v2.png")

# Feature contribution

## SHAP — configuration comparison

How does feature importance shift when moving from single-level to integrated?
- **Option A** (side-by-side bars): micro-only | integrated | macro-only per model — shows which level dominates in integrated
- **Option C** (scatter): single-level importance vs integrated importance — above diagonal = amplified, below = crowded out

## NB11a

### Top features

In [ ]:
# A55: short labels under their own name; redefining MODEL_LABELS made earlier cells depend on execution order
# Option A: side-by-side SHAP bars
# Three panels per model: micro-only | integrated | macro-only.
# Each panel has its own x-axis scale (models differ greatly in SHAP magnitude).
# Values shown inside each bar in white; black used when the bar is too short
# to read white text against the background.

import matplotlib.patches as mpatches

df_shap = pd.read_parquet(SHAP_FILES['11a'])
df_shap = df_shap[df_shap['variant'] == 'baseline_t1'].copy()

all_shap_cols = [c for c in df_shap.columns if c.startswith('shap_')]

micro_shap = [c for c in all_shap_cols if c[5:] not in MACRO_FEATURES]
macro_shap = [c for c in all_shap_cols if c not in micro_shap]

NAME_MAP = {
    'shap_r_cti':          'Cost-to-income ratio',
    'shap_x_loans_gross':  'Gross loans & advances',
    'shap_x_ib_liab':      'Interbank liabilities',
    'shap_inc_nii':        'Net interest income',
    'shap_p_nim':          'Net interest margin',
    'shap_inc_op':         'Operating income',
    'shap_p_roa':          'ROA',
    'shap_p_roe':          'ROE',
    'shap_x_dep':          'Customer deposits',
    'shap_c_eqratio':      'Equity ratio',
    'shap_l_assets':       'Log total assets',
    'shap_l_loans':        'Log loans',
    'shap_r_loans_assets': 'Loans to assets',
    'shap_r_dep_liab':     'Deposits to liabilities',
    'shap_c_leverage':     'Leverage (assets/equity)',
    'shap_NGDP_RPCH':      'Real GDP growth',
    'shap_NGDPD':          'Nominal GDP',
    'shap_PCPIPCH':        'Inflation (CPI)',
    'shap_LUR':            'Unemployment rate',
    'shap_BCA_NGDPD':      'Current account (% GDP)',
    'shap_BIS_REER':       'Real eff. exchange rate',
    'shap_RXF11FX_REVS':   'Foreign exchange reserves',
    'shap_reer_g':         'REER growth',
    'shap_fx_g':           'FX reserves growth',
}

MODEL_ORDER  = ['logistic_regression', 'decision_tree', 'random_forest',
                'xgboost', 'lightgbm', 'mlp']
MODEL_LABELS_SHORT = {'logistic_regression': 'LR', 'decision_tree': 'DT',
                'random_forest': 'RF', 'xgboost': 'XGBoost',
                'lightgbm': 'LightGBM', 'mlp': 'MLP'}

MICRO_COLOR = '#1f77b4'
MACRO_COLOR = '#ff7f0e'
TOP_N = 10
# A bar narrower than this fraction of the panel's x-range is too short for
# white inside text -- fall back to black text just outside the bar tip
MIN_BAR_FRACTION = 0.18

def mean_abs_shap(sub, cols):
    return sub[cols].abs().mean(skipna=True)

def get_importance(df, feature_set, model, cols):
    mask = (df['feature_set'] == feature_set) & (df['model_name'] == model)
    imp  = mean_abs_shap(df[mask], cols)
    return imp[imp.notna() & (imp > 0)].sort_values(ascending=False)

fig, axes = plt.subplots(6, 3, figsize=(18, 22))
#fig.suptitle(
#    'Top features per configuration\n',
#    
#    #'Note: x-axis scale differs per model row',
#    fontsize=13, y=1.01, fontweight='bold'
#)

configs = [
    ('micro',       micro_shap,    MICRO_COLOR, 'Micro-only'),
    ('integrated',  all_shap_cols, None,        'Integrated'),
    ('macro',       macro_shap,    MACRO_COLOR, 'Macro-only'),
]

for row_i, model in enumerate(MODEL_ORDER):
    for col_i, (fs, cols, color, title) in enumerate(configs):
        ax  = axes[row_i][col_i]
        imp = get_importance(df_shap, fs, model, cols).head(TOP_N)

        bar_colors = ([MICRO_COLOR if c in micro_shap else MACRO_COLOR for c in imp.index]
                      if fs == 'integrated' else color)

        ax.barh(range(len(imp)), imp.values, color=bar_colors)
        ax.set_yticks(range(len(imp)))
        ax.set_yticklabels([NAME_MAP.get(c, c) for c in imp.index], fontsize=7.5)
        ax.invert_yaxis()

        x_max = ax.get_xlim()[1]
        for i, val in enumerate(imp.values):
            if val / x_max >= MIN_BAR_FRACTION:
                # Bar is wide enough: white text centred inside
                ax.text(val / 2, i, f'{val:.4f}',
                        va='center', ha='center',
                        fontsize=6.5, color='white', fontweight='bold')
            else:
                # Bar too narrow: black text just outside the tip
                ax.text(val + x_max * 0.01, i, f'{val:.4f}',
                        va='center', ha='left',
                        fontsize=6.5, color='#333333')

        ax.set_xlabel('Mean |SHAP|', fontsize=8)
        if col_i == 0:
            ax.set_ylabel(MODEL_LABELS_SHORT[model], fontsize=10, fontweight='bold')
        if row_i == 0:
            ax.set_title(title, fontsize=9)

micro_patch = mpatches.Patch(color=MICRO_COLOR, label='Micro feature')
macro_patch = mpatches.Patch(color=MACRO_COLOR, label='Macro feature')
fig.legend(handles=[micro_patch, macro_patch], loc='lower center',
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.savefig(OUTPUT / 'shap_config_comparison_A.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved --> shap_config_comparison_A.png')

### Relative contribution

In [ ]:
# ── Option C: scatter — single-level importance vs integrated importance ───────
# Each dot is one feature. x = importance in its own single-level model;
# y = importance in the integrated model.
# Above the dashed diagonal → amplified in integrated; below → crowded out.

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    'Option C — Single-level importance vs integrated importance\n'
    '(above diagonal = amplified; below = crowded out; baseline_t1, NB11a)',
    fontsize=13,
)
axes = axes.flatten()

# Only plot features with meaningful importance (avoids bottom-left noise cluster)
MIN_IMPORTANCE = 0.001

for ax_i, model in enumerate(MODEL_ORDER):
    ax = axes[ax_i]

    int_imp = get_importance(df_shap, 'integrated', model, all_shap_cols)
    mic_imp = get_importance(df_shap, 'micro',      model, micro_shap)
    mac_imp = get_importance(df_shap, 'macro',      model, macro_shap)

    for feat, color, sl_imp in (
        [(f, MICRO_COLOR, mic_imp) for f in micro_shap] +
        [(f, MACRO_COLOR, mac_imp) for f in macro_shap]
    ):
        sl_val  = sl_imp.get(feat, 0)
        int_val = int_imp.get(feat, 0)
        # Skip near-zero features that just add clutter to the bottom-left
        if sl_val < MIN_IMPORTANCE and int_val < MIN_IMPORTANCE:
            continue
        ax.scatter(sl_val, int_val, color=color, alpha=0.75, s=40, zorder=3)
        if feat in int_imp.head(TOP_N).index or sl_val > MIN_IMPORTANCE * 5:
            ax.annotate(NAME_MAP.get(feat, feat), (sl_val, int_val),
                        fontsize=6.5, xytext=(3, 2), textcoords='offset points',
                        color=color)

    lim = max(ax.get_xlim()[1], ax.get_ylim()[1]) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=0.8, alpha=0.4, zorder=1)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.set_xlabel('Mean |SHAP| (single-level)', fontsize=8)
    ax.set_ylabel('Mean |SHAP| (integrated)',   fontsize=8)
    ax.set_title(MODEL_LABELS_SHORT[model], fontsize=10, fontweight='bold')

micro_patch = mpatches.Patch(color=MICRO_COLOR, label='Micro feature')
macro_patch = mpatches.Patch(color=MACRO_COLOR, label='Macro feature')
fig.legend(handles=[micro_patch, macro_patch], loc='lower center',
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.savefig(OUTPUT / 'shap_config_comparison_C.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → shap_config_comparison_C.png')

In [ ]:
# ── Option C (ratio variant) — integrated/single-level importance ratio ────────
# y-axis: ratio of integrated importance to single-level importance.
# Ratio > 1 = gained importance when integrated; < 1 = crowded out; = 1 = no change.
# x-axis: single-level importance — so you can see whether large or small features
# are the ones being displaced.
# This normalises out the scale difference between micro-only (15 features) and
# macro-only (9 features), making micro and macro dots directly comparable.

MIN_SL = 0.002   # minimum single-level importance to compute a stable ratio

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
#fig.suptitle(
#    'Relative contribution change — integrated / single-level importance ratio\n',
#    fontsize=13,
#)
axes = axes.flatten()

for ax_i, model in enumerate(MODEL_ORDER):
    ax = axes[ax_i]

    int_imp = get_importance(df_shap, 'integrated', model, all_shap_cols)
    mic_imp = get_importance(df_shap, 'micro',      model, micro_shap)
    mac_imp = get_importance(df_shap, 'macro',      model, macro_shap)

    for feat, color, sl_imp in (
        [(f, MICRO_COLOR, mic_imp) for f in micro_shap] +
        [(f, MACRO_COLOR, mac_imp) for f in macro_shap]
    ):
        sl_val  = sl_imp.get(feat, 0)
        int_val = int_imp.get(feat, 0)
        # Skip features with near-zero single-level importance — ratio is unstable
        if sl_val < MIN_SL:
            continue
        ratio = int_val / sl_val
        ax.scatter(sl_val, ratio, color=color, alpha=0.75, s=50, zorder=3)
        ax.annotate(NAME_MAP.get(feat, feat), (sl_val, ratio),
                    fontsize=6.5, xytext=(3, 2), textcoords='offset points',
                    color=color)

    # Horizontal reference line at ratio = 1 (no change)
    ax.axhline(1, color='black', lw=0.8, linestyle='--', alpha=0.4, zorder=1)
    ax.set_xlim(left=0)
    ax.set_xlabel('Mean |SHAP| (single-level)', fontsize=8)
    ax.set_ylabel('Integrated / single-level ratio', fontsize=8)
    ax.set_title(MODEL_LABELS_SHORT[model], fontsize=10, fontweight='bold')

micro_patch = mpatches.Patch(color=MICRO_COLOR, label='Micro feature')
macro_patch = mpatches.Patch(color=MACRO_COLOR, label='Macro feature')
fig.legend(handles=[micro_patch, macro_patch], loc='lower center',
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.savefig(OUTPUT / 'shap_config_comparison_C_ratio.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → shap_config_comparison_C_ratio.png')
# -- Split into two vertical 3x1 figures for document pasting --
for part_models, suffix in [
    (MODEL_ORDER[:3], 'p1'),
    (MODEL_ORDER[3:], 'p2'),
]:
    fig2, axes2 = plt.subplots(3, 1, figsize=(7, 15))
    for ax_i, model in enumerate(part_models):
        ax = axes2[ax_i]
        int_imp = get_importance(df_shap, 'integrated', model, all_shap_cols)
        mic_imp = get_importance(df_shap, 'micro',      model, micro_shap)
        mac_imp = get_importance(df_shap, 'macro',      model, macro_shap)
        for feat, color, sl_imp in (
            [(f, MICRO_COLOR, mic_imp) for f in micro_shap] +
            [(f, MACRO_COLOR, mac_imp) for f in macro_shap]
        ):
            sl_val  = sl_imp.get(feat, 0)
            int_val = int_imp.get(feat, 0)
            if sl_val < MIN_SL:
                continue
            ratio = int_val / sl_val
            ax.scatter(sl_val, ratio, color=color, alpha=0.75, s=50, zorder=3)
            ax.annotate(NAME_MAP.get(feat, feat), (sl_val, ratio),
                        fontsize=6.5, xytext=(3, 2), textcoords='offset points',
                        color=color)
        ax.axhline(1, color='black', lw=0.8, linestyle='--', alpha=0.4, zorder=1)
        ax.set_xlim(left=0)
        ax.set_xlabel('Mean |SHAP| (single-level)', fontsize=8)
        ax.set_ylabel('Integrated / single-level ratio', fontsize=8)
        ax.set_title(MODEL_LABELS_SHORT[model], fontsize=10, fontweight='bold')
    micro_patch = mpatches.Patch(color=MICRO_COLOR, label='Micro feature')
    macro_patch = mpatches.Patch(color=MACRO_COLOR, label='Macro feature')
    fig2.legend(handles=[micro_patch, macro_patch], loc='lower center',
                ncol=2, fontsize=10, bbox_to_anchor=(0.5, 0.01))
    plt.tight_layout()
    out_name = f'shap_config_comparison_C_ratio_{suffix}.png'
    fig2.savefig(OUTPUT / out_name, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved --> {out_name}')

In [ ]:
# Option C (ratio, normalised x) -- same as C_ratio but x-axis shows each feature's
# *share* of total single-level explanation rather than absolute mean |SHAP|.
# Dividing by the sum of all mean |SHAP| values in that config puts micro and macro
# features on the same 0-1 scale, so x-positions are directly comparable across levels.
# y-axis (ratio) is unchanged: integrated absolute importance / single-level absolute importance.

MIN_SL = 0.002

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    'Relative contribution change — integrated / single-level importance ratio\n',
    fontsize=13, fontweight='bold',
)
axes = axes.flatten()

for ax_i, model in enumerate(MODEL_ORDER):
    ax = axes[ax_i]

    int_imp = get_importance(df_shap, 'integrated', model, all_shap_cols)
    mic_imp = get_importance(df_shap, 'micro',      model, micro_shap)
    mac_imp = get_importance(df_shap, 'macro',      model, macro_shap)

    # Normalise: each feature's importance as a share of its single-level total
    # so micro (15 features) and macro (9 features) are on the same 0-1 scale
    mic_total = mic_imp.sum() if mic_imp.sum() > 0 else 1
    mac_total = mac_imp.sum() if mac_imp.sum() > 0 else 1

    for feat, color, sl_imp, sl_total in (
        [(f, MICRO_COLOR, mic_imp, mic_total) for f in micro_shap] +
        [(f, MACRO_COLOR, mac_imp, mac_total) for f in macro_shap]
    ):
        sl_val  = sl_imp.get(feat, 0)
        int_val = int_imp.get(feat, 0)
        if sl_val < MIN_SL:
            continue
        x     = sl_val / sl_total   # normalised share (0-1), comparable across levels
        ratio = int_val / sl_val    # ratio unchanged
        ax.scatter(x, ratio, color=color, alpha=0.75, s=50, zorder=3)
        ax.annotate(NAME_MAP.get(feat, feat), (x, ratio),
                    fontsize=6.5, xytext=(3, 2), textcoords='offset points',
                    color=color)

    ax.axhline(1, color='black', lw=0.8, linestyle='--', alpha=0.4, zorder=1)
    ax.set_xlim(left=0)
    ax.set_xlabel('Share of total single-level explanation', fontsize=8)
    ax.set_ylabel('Integrated / single-level ratio', fontsize=8)
    ax.set_title(MODEL_LABELS_SHORT[model], fontsize=10, fontweight='bold')

micro_patch = mpatches.Patch(color=MICRO_COLOR, label='Micro feature')
macro_patch = mpatches.Patch(color=MACRO_COLOR, label='Macro feature')
fig.legend(handles=[micro_patch, macro_patch], loc='lower center',
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.savefig(OUTPUT / 'shap_config_comparison_C_ratio_norm.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved --> shap_config_comparison_C_ratio_norm.png')

## Cross-specification feature contribution (Discussion / RQ3)

Tracks how the **integrated** model's feature importance shifts when the indicator set
is extended (baseline -> extended), pooled by **model class** (Tree / Linear / Neural).
This is the one contribution view not already in Section 4.3: Figure 9 shows the
single-level -> integrated shift, Figure 10 the per-config ranking *within* one spec.
Here the axis is the *specification* itself, isolating where the credit-cycle features
(credit gap, credit growth, NFA growth) enter and what they displace.

- **Bump charts** (Discussion figures): rank movement, one per lag span.
- **Heatmaps** (appendix): the underlying share magnitudes.

Exports to `outputs/figures/results_discussion/`.

In [ ]:
# Cross-specification integrated feature-importance — pooled by model class.
# Built from the SHAP parquets (SHAP_FILES) for the integrated config only.
import matplotlib.patches as mpatches

OUTPUT_DISC = BASE / 'outputs/figures' / 'results_discussion'
OUTPUT_DISC.mkdir(parents=True, exist_ok=True)

CREDIT_COLOR = '#d62728'  # red — highlight the credit-cycle features that the extended set adds
# Pool the four tree learners into one class; LR and MLP stand alone. Model choice is only
# interpretable at the CLASS level here (we can read a pattern across trees, not motivate why
# one individual learner picks a given feature), so pooling is the honest granularity.
CLASS = {'decision_tree': 'Tree', 'random_forest': 'Tree', 'xgboost': 'Tree',
         'lightgbm': 'Tree', 'logistic_regression': 'Linear (LR)', 'mlp': 'Neural (MLP)'}
CLASS_ORDER = ['Tree', 'Linear (LR)', 'Neural (MLP)']

# Extend the Section 4.3 NAME_MAP with the extended-set feature codes (no shap_ prefix here).
FI_NAMES = {
    'r_cti': 'Cost-to-income', 'x_loans_gross': 'Gross loans', 'x_ib_liab': 'Interbank liab.',
    'inc_nii': 'Net interest income', 'p_nim': 'Net interest margin', 'inc_op': 'Operating income',
    'p_roa': 'ROA', 'p_roe': 'ROE', 'x_dep': 'Customer deposits', 'c_eqratio': 'Equity ratio',
    'l_assets': 'Log total assets', 'l_loans': 'Log loans', 'r_loans_assets': 'Loans/assets',
    'r_dep_liab': 'Deposits/liab.', 'c_leverage': 'Leverage', 'r_loans_dep': 'Loans/deposits',
    'l_obsi': 'Log off-bal. items', 'cr_impchg': 'Impairment charges',
    'NGDP_RPCH': 'Real GDP growth', 'NGDPD': 'Nominal GDP', 'PCPIPCH': 'Inflation (CPI)',
    'LUR': 'Unemployment', 'BCA_NGDPD': 'Current acc. (%GDP)', 'BIS_REER': 'REER (level)',
    'RXF11FX_REVS': 'FX reserves', 'reer_g': 'REER growth', 'fx_g': 'FX reserves growth',
    'credit_gap': 'Credit-to-GDP gap', 'credit_g': 'Credit growth', 'nfa_gdp': 'NFA change (%GDP)',
    'credit_gap_missing': 'Credit gap unavailable', 'FS.AST.PRVT.GD.ZS': 'Private credit (%GDP)',
}
CREDIT_FEATS = {'credit_gap', 'credit_gap_missing', 'credit_g', 'nfa_gdp', 'FS.AST.PRVT.GD.ZS'}  # introduced by the extended set
N_SHOW = 10  # top features displayed per column
# Shorter labels for the bump only — long names bleed into the neighbouring panel.
BUMP_SHORT = {'NGDP_RPCH': 'Real GDP gr.', 'NGDPD': 'Nominal GDP', 'credit_gap': 'Credit gap',
              'credit_g': 'Credit growth', 'nfa_gdp': 'NFA change', 'credit_gap_missing': 'Gap unavail.', 'FS.AST.PRVT.GD.ZS': 'Priv. credit',
              'p_nim': 'Net int. margin', 'inc_nii': 'Net int. income', 'BCA_NGDPD': 'Curr. acc.',
              'RXF11FX_REVS': 'FX reserves', 'fx_g': 'FX res. growth', 'x_dep': 'Cust. deposits',
              'BIS_REER': 'REER', 'r_loans_assets': 'Loans/assets', 'reer_g': 'REER growth'}

def _is_micro(f):
    return f not in MACRO_FEATURES

def _feat_color(f):
    return CREDIT_COLOR if f in CREDIT_FEATS else (MICRO_COLOR if _is_micro(f) else MACRO_COLOR)

def pooled_shares(spec):
    """Within-model normalised |SHAP| share for the integrated config, pooled by class.
    Raw |SHAP| magnitudes differ by orders of magnitude across model types, so each model's
    importances are first turned into shares (sum to 1) before averaging within a class —
    otherwise the largest-magnitude model would dominate the pooled ranking."""
    df = pd.read_parquet(SHAP_FILES[spec])
    df = df[(df['variant'] == 'baseline_t1') & (df['feature_set'] == 'integrated')]
    cols = [c for c in df.columns if c.startswith('shap_')]
    rows = []
    for m, sub in df.groupby('model_name'):
        imp = sub[cols].abs().mean()
        imp = imp[imp > 0]
        share = imp / imp.sum()
        for c, s in share.items():
            rows.append({'class': CLASS[m], 'feat': c[5:], 'share': s})
    return pd.DataFrame(rows).groupby(['class', 'feat'])['share'].mean().reset_index()

def build_bump(spec_b, spec_e, lag_label, fname):
    """Slopegraph of the top-N integrated features per class, baseline -> extended.
    Each column ranked independently; lines connect features present in BOTH columns' top-N,
    lone dots mark features that enter ('(new)' if absent from the baseline set, '(↑)' if it
    rose from below top-N) or drop out ('(↓)', faded)."""
    base, ext = pooled_shares(spec_b), pooled_shares(spec_e)
    base_feats = set(base['feat'])
    fig, axes = plt.subplots(1, 3, figsize=(21, 8.5), gridspec_kw={'wspace': 1.25})
    for ax, cls in zip(axes, CLASS_ORDER):
        rb = base[base['class'] == cls].sort_values('share', ascending=False).reset_index(drop=True)
        re = ext[ext['class'] == cls].sort_values('share', ascending=False).reset_index(drop=True)
        topb = {r['feat']: i + 1 for i, r in rb.head(N_SHOW).iterrows()}
        tope = {r['feat']: i + 1 for i, r in re.head(N_SHOW).iterrows()}
        for feat in set(topb) | set(tope):
            col, lbl = _feat_color(feat), BUMP_SHORT.get(feat, FI_NAMES.get(feat, feat))
            if feat in topb and feat in tope:
                ax.plot([0, 1.6], [topb[feat], tope[feat]], '-o', color=col, lw=2, ms=6, alpha=0.85)
                ax.text(-0.13, topb[feat], lbl, ha='right', va='center', fontsize=6.5, color=col)
                ax.text(1.73, tope[feat], lbl, ha='left', va='center', fontsize=6.5, color=col)
            elif feat in tope:
                tag = ' (new)' if feat not in base_feats else ' (↑)'
                ax.plot([1.6], [tope[feat]], 'o', color=col, ms=8)
                ax.text(1.73, tope[feat], lbl + tag, ha='left', va='center', fontsize=6.5,
                        color=col, fontweight='bold' if tag == ' (new)' else 'normal')
            else:
                ax.plot([0], [topb[feat]], 'o', color=col, ms=6, alpha=0.45)
                ax.text(-0.13, topb[feat], lbl + ' (↓)', ha='right', va='center',
                        fontsize=6.5, color=col, alpha=0.6)
        ax.set_title(cls, fontsize=11, fontweight='bold')
        ax.set_xlim(-2.0, 3.4); ax.set_xticks([0, 1.6])
        ax.set_xticklabels(['Baseline', 'Extended'], fontsize=9)
        ax.set_ylim(N_SHOW + 0.5, 0.5); ax.set_yticks(range(1, N_SHOW + 1))
        if cls == CLASS_ORDER[0]:
            ax.set_ylabel('Rank within integrated config', fontsize=6.5)
        for sp in ['top', 'right']:
            ax.spines[sp].set_visible(False)
    handles = [mpatches.Patch(color=MICRO_COLOR, label='Micro'),
               mpatches.Patch(color=MACRO_COLOR, label='Macro'),
               mpatches.Patch(color=CREDIT_COLOR, label='Credit-cycle (extended set)')]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(f'Integrated feature-importance rank: Baseline → Extended set ({lag_label}), '
                 f'pooled by model class', fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0.09, 1, 0.95])
    plt.savefig(OUTPUT_DISC / fname, bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved -->', fname)

def build_heatmap(spec_b, spec_e, lag_label, fname):
    """Appendix: share magnitudes behind the bump chart. Rows = features (micro block then
    macro block, each sorted by extended share); columns = Baseline / Extended; per class."""
    allsh = pd.concat([pooled_shares(spec_b).assign(spec='Baseline'),
                       pooled_shares(spec_e).assign(spec='Extended')], ignore_index=True)
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 9), gridspec_kw={'wspace': 0.75})
    im = None
    for ax, cls in zip(axes, CLASS_ORDER):
        piv = (allsh[allsh['class'] == cls]
               .pivot_table(index='feat', columns='spec', values='share', fill_value=0)
               .reindex(columns=['Baseline', 'Extended']))
        piv['ismic'] = [0 if _is_micro(f) else 1 for f in piv.index]
        piv = piv.sort_values(['ismic', 'Extended'], ascending=[True, False]).drop(columns='ismic')
        im = ax.imshow(piv.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=0.28)
        ax.set_xticks([0, 1]); ax.set_xticklabels(['Baseline', 'Extended'], fontsize=9)
        ax.set_yticks(range(len(piv)))
        ax.set_yticklabels([FI_NAMES.get(f, f) + (' *' if f in CREDIT_FEATS else '')
                            for f in piv.index], fontsize=7)
        for tick, f in zip(ax.get_yticklabels(), piv.index):
            tick.set_color(_feat_color(f))
        ax.set_title(cls, fontsize=11, fontweight='bold')
        ax.set_xlim(-0.55, 1.55)
        for i in range(len(piv)):
            for j in range(2):
                v = piv.values[i, j]
                if v > 0:
                    ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6,
                            color='black' if v < 0.18 else 'white')
    fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label='Mean share of |SHAP| (integrated)')
    #fig.suptitle(f'Integrated feature contribution: Baseline vs Extended set ({lag_label})',
    #             #f'pooled by model class\n* = credit-cycle feature introduced by the extended set',
    #             fontsize=11, fontweight='bold')
    plt.savefig(OUTPUT_DISC / fname, bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved -->', fname)

# lag-1 (11a -> 11b): the indicator-set extension at the main horizon (credit_gap enters)
build_bump('11a', '11b', 'lag-1', 'fi_crossspec_bump_lag1.png')
build_heatmap('11a', '11b', 'lag-1', 'fi_crossspec_heatmap_lag1.png')
# lag-2 (11c -> 11d): same extension at the 2-year horizon — horizon-robustness check
build_bump('11c', '11d', 'lag-2', 'fi_crossspec_bump_lag2.png')
build_heatmap('11c', '11d', 'lag-2', 'fi_crossspec_heatmap_lag2.png')

### Feature direction — which way does each feature push crisis probability?

Companion to the SHAP importance figures above. Those use mean |SHAP|, which answers *how much*
a feature matters but discards the sign; this table recovers *which way* it pushes, so the top
features can be reasoned about economically. Exported to `outputs/tables/` as `direction_<spec>`.

In [ ]:
# Feature DIRECTION table - which way does each feature push crisis probability?
#
# WHY: every SHAP figure above aggregates mean |SHAP|, which answers "how much does this
# feature matter" but deliberately throws away the sign. To reason economically about the
# top features we also need the direction. The signed per-observation SHAP values are
# already stored in shap_values_*.parquet, so this needs no model re-run.
#
# METHOD: correlate each feature's VALUE against its own SHAP value, using Spearman.
#   rho > 0  -> higher values of the feature push crisis probability UP
#   rho < 0  -> higher values push it DOWN
# Spearman (rank-based) rather than Pearson because it is unaffected by the StandardScaler
# applied for LR/MLP, and because it imposes no linear value->contribution form.
#
# A rank correlation ALONE is not enough: it can be large while the underlying shape
# reverses. LUR is the case in point - SHAP is positive at low unemployment, turns negative
# above ~8-9%, and still yields rho = -0.55, which would read as a clean "higher unemployment
# lowers crisis risk" if taken at face value. So each feature is also binned into quintiles
# and its mean-SHAP profile checked for a reversal; a direction is only reported when it is
# strong, agreed across models AND monotone.
#
# NOTE: mean SIGNED SHAP is deliberately not used - SHAP values are centred on the sample's
# average prediction, so a plain mean sits near zero regardless of the true direction.

PANEL_FILES = {
    '11a': BASE / 'data/engineered/parquet_saves/me_merged_base.parquet',
    '11b': BASE / 'data/engineered/parquet_saves/me_merged_robust.parquet',
    '11c': BASE / 'data/engineered/parquet_saves/me_merged_base_lag2.parquet',
    '11d': BASE / 'data/engineered/parquet_saves/me_merged_robust_lag2.parquet',
}
SPEC_LABEL = {'11a': 'Baseline / lag-1', '11b': 'Extended / lag-1',
              '11c': 'Baseline / lag-2', '11d': 'Extended / lag-2'}

# Classify by membership in the full macro vocabulary rather than a per-spec list: the four
# specs carry different macro subsets, and a hardcoded list per spec silently mislabels a
# feature as micro the moment a spec's macro set changes.
MACRO_ALL = {
    'NGDP_RPCH', 'NGDPD', 'PCPIPCH', 'LUR', 'BCA_NGDPD', 'BIS_REER', 'RXF11FX_REVS',
    'reer_g', 'fx_g', 'FS.AST.PRVT.GD.ZS', 'FM.AST.NFRG.CN', 'nfa_gdp', 'credit_g', 'credit_gap', 'credit_gap_missing',  # A23/A24
}

# Publication labels for the appendix table - check against Appendix B1/B2 before pasting.
FEATURE_LABELS = {
    'r_cti': 'Cost-to-income ratio', 'x_loans_gross': 'Gross loans',
    'x_ib_liab': 'Interbank liabilities', 'cr_impchg': 'Impairment charges',
    'inc_nii': 'Net interest income', 'p_nim': 'Net interest margin',
    'inc_op': 'Operating income', 'p_roa': 'Return on assets', 'p_roe': 'Return on equity',
    'x_dep': 'Customer deposits', 'c_eqratio': 'Equity ratio', 'l_assets': 'Log total assets',
    'l_loans': 'Log loans', 'r_loans_assets': 'Loans-to-assets',
    'r_dep_liab': 'Deposits-to-liabilities', 'r_loans_dep': 'Loans-to-deposits',
    'c_leverage': 'Leverage', 'l_obsi': 'Log off-balance-sheet items',
    'NGDP_RPCH': 'Real GDP growth', 'NGDPD': 'Nominal GDP (log)', 'PCPIPCH': 'Inflation (CPI)',
    'LUR': 'Unemployment rate', 'BCA_NGDPD': 'Current account (% GDP)',
    'BIS_REER': 'Real effective exchange rate', 'RXF11FX_REVS': 'FX reserves (log)',
    'reer_g': 'REER growth', 'fx_g': 'FX reserves growth',
    'FS.AST.PRVT.GD.ZS': 'Private credit (% GDP)', 'FM.AST.NFRG.CN': 'Net foreign assets',
    'nfa_gdp': 'Change in net foreign assets (% GDP)', 'credit_g': 'Credit growth',
    'credit_gap': 'Credit-to-GDP gap', 'credit_gap_missing': 'Credit gap unavailable (0/1)',
}
VERDICT_WORDS = {'reportable': 'Consistent',
                 'non-monotone': 'Sign consistent, shape reverses',
                 'weak/noisy': 'No clear direction'}

MODEL_SHORT = {'logistic_regression': 'LR', 'decision_tree': 'DT', 'random_forest': 'RF',
               'xgboost': 'XGB', 'lightgbm': 'LGBM', 'mlp': 'MLP'}


def _spearman(x, y):
    """Pearson correlation on the ranks - avoids a scipy dependency."""
    xr, yr = pd.Series(x).rank().to_numpy(), pd.Series(y).rank().to_numpy()
    return np.nan if xr.std() == 0 or yr.std() == 0 else float(np.corrcoef(xr, yr)[0, 1])


def _direction(g):
    m = g['value'].notna() & g['shap'].notna()
    if m.sum() < 30 or g.loc[m, 'value'].nunique() < 3:
        return np.nan
    return _spearman(g.loc[m, 'value'], g.loc[m, 'shap'])


def _monotone(g, n_bins=5, tol=0.10):
    """1.0 if the binned value->contribution profile never reverses, else 0.0.

    `tol` ignores reversals smaller than 10% of the profile's range so trivial wiggles do
    not flag an otherwise clean feature.
    """
    m = g['value'].notna() & g['shap'].notna()
    if m.sum() < 50 or g.loc[m, 'shap'].nunique() < 2:
        return np.nan
    gg = g.loc[m]
    try:
        prof = gg.groupby(pd.qcut(gg['value'], n_bins, duplicates='drop'),
                          observed=True)['shap'].mean().to_numpy()
    except ValueError:
        return np.nan
    if len(prof) < 3 or prof.max() == prof.min():
        return np.nan
    steps = np.diff(prof)
    big = steps[np.abs(steps) > tol * (prof.max() - prof.min())]
    return float(len(set(np.sign(big))) <= 1)


def build_direction_table(spec, variant='baseline_t1', config='integrated'):
    """Signed direction of every feature's SHAP contribution, per model, for one spec."""
    sv = pd.read_parquet(SHAP_FILES[spec])
    sv = sv[(sv['variant'] == variant) & (sv['feature_set'] == config)].copy()
    shap_cols = [c for c in sv.columns if c.startswith('shap_')]
    features = [c[5:] for c in shap_cols]

    panel = pd.read_parquet(PANEL_FILES[spec])[['bank_id', 'year'] + features]

    # Melt each side separately and join on (bank_id, year, feature): a wide-to-wide merge
    # followed by two melts is far easier to get silently wrong.
    sv_long = sv.melt(id_vars=['bank_id', 'year', 'test_year', 'model_name'],
                      value_vars=shap_cols, var_name='feature', value_name='shap')
    sv_long['feature'] = sv_long['feature'].str[5:]
    val_long = panel.melt(id_vars=['bank_id', 'year'], value_vars=features,
                          var_name='feature', value_name='value')
    df_dir = sv_long.merge(val_long, on=['bank_id', 'year', 'feature'], how='left')

    grp = df_dir.groupby(['feature', 'model_name'], observed=True)
    rho = grp.apply(_direction, include_groups=False).unstack('model_name')
    mono = grp.apply(_monotone, include_groups=False).unstack('model_name')

    order = [m for m in MODEL_ORDER if m in rho.columns]
    rho = rho[order].rename(columns=MODEL_SHORT)
    mono = mono[order].rename(columns=MODEL_SHORT).reindex(rho.index)

    signs = np.sign(rho)
    agree = np.maximum((signs > 0).sum(axis=1), (signs < 0).sum(axis=1))
    median_rho = rho.median(axis=1)
    n_mono, n_eval = mono.sum(axis=1), mono.notna().sum(axis=1)

    # Report a direction only when all three conditions hold; otherwise say why not.
    strong, agreed = median_rho.abs() > 0.30, agree >= 5
    shaped = (n_mono / n_eval.replace(0, np.nan)) >= 0.80
    verdict = np.where(strong & agreed & shaped, 'reportable',
               np.where(strong & agreed & ~shaped, 'non-monotone', 'weak/noisy'))

    out = pd.DataFrame({
        'level': ['macro' if f in MACRO_ALL else 'micro' for f in rho.index],
        'mean_abs_shap': df_dir.groupby('feature', observed=True)['shap']
                               .apply(lambda s: s.abs().mean()).reindex(rho.index),
        'median_rho': median_rho,
        'sign_agree': [f'{a}/{len(order)}' for a in agree],
        'monotone': [f'{int(a)}/{int(b)}' if b else '-' for a, b in zip(n_mono, n_eval)],
        'direction': np.where(median_rho > 0, 'raises risk', 'lowers risk'),
        'verdict': verdict,
    }).join(rho)
    return out.sort_values('mean_abs_shap', ascending=False)


directions = {s: build_direction_table(s) for s in ['11a', '11b', '11c', '11d']}
direction_11a, direction_11b = directions['11a'], directions['11b']

print('Feature direction - NB11a baseline, integrated config')
print('rho = Spearman(feature value, its SHAP value); + = higher value raises crisis probability')
print('verdict: reportable = direction can be stated | non-monotone = shape reverses, state the '
      'shape not a sign | weak/noisy = no readable direction\n')
display(direction_11a.round(3))

# Append to the existing workbooks rather than writing a new file, keeping one workbook per spec.
for spec, tbl in directions.items():
    wb = 'results_15_11a.xlsx' if spec == '11a' else 'results_15_robust.xlsx'
    path = OUT / wb
    kw = dict(mode='a', if_sheet_exists='replace') if path.exists() else dict(mode='w')
    with pd.ExcelWriter(path, engine='openpyxl', **kw) as writer:
        tbl.to_excel(writer, sheet_name=f'direction_{spec}')
print(f'Saved direction sheets for {list(directions)} into {OUT}')


def appendix_table(spec):
    """Publication-ready version for the appendix: readable labels, no jargon, and no
    direction printed where the evidence does not support one."""
    t = directions[spec]
    arrow = {'raises risk': 'Higher \u2192 higher risk', 'lowers risk': 'Higher \u2192 lower risk'}
    return pd.DataFrame({
        'Variable': [FEATURE_LABELS.get(f, f) for f in t.index],
        'Code': t.index,
        'Level': t['level'].str.capitalize(),
        'Mean |SHAP|': t['mean_abs_shap'].round(3),
        'Direction': [arrow[d] if v != 'weak/noisy' else '\u2014'
                      for d, v in zip(t['direction'], t['verdict'])],
        'rho': t['median_rho'].round(2),
        'Models agreeing': t['sign_agree'],
        'Monotone': t['monotone'],
        'Evidence': [VERDICT_WORDS[v] for v in t['verdict']],
    }).reset_index(drop=True)


# Appendix H companion table (magnitude lives in H; this is the sign). Lag-1 specs only -
# the lag-2 specs are used below as a stability check rather than as further tables.
# CSV for archiving, xlsx for pasting straight into the manuscript.
with pd.ExcelWriter(OUT / 'appendix_direction.xlsx', engine='openpyxl') as writer:
    for spec in ['11a', '11b']:
        tbl = appendix_table(spec)
        tbl.to_csv(OUT / f'appendix_direction_{spec}.csv', index=False, encoding='utf-8-sig')
        tbl.to_excel(writer, sheet_name=SPEC_LABEL[spec].replace(' / ', ' '), index=False)
print(f"Saved -> {OUT / 'appendix_direction.xlsx'} (+ per-spec CSVs)")

# Horizon stability: of the features whose direction is established at lag-1, how many keep
# the same sign at lag-2? Feeds the cross-model/cross-horizon stability point (feedback #4).
for lag1, lag2, name in [('11a', '11c', 'Baseline'), ('11b', '11d', 'Extended')]:
    a, b = directions[lag1], directions[lag2]
    established = a.index[a['verdict'] != 'weak/noisy']
    shared = [f for f in established if f in b.index]
    same = [f for f in shared if np.sign(a.loc[f, 'median_rho']) == np.sign(b.loc[f, 'median_rho'])]
    flipped = [f for f in shared if f not in same]
    print(f'\n{name}: {len(same)}/{len(shared)} directions hold from lag-1 to lag-2'
          f'{" | flips: " + ", ".join(flipped) if flipped else ""}')

### SHAP share displacement — what the extended indicator set actually bought (Discussion)

Supports the Discussion's extension-asymmetry argument. SHAP shares sum to one by construction,
so whatever the newly added variables capture must come out of the incumbents' share. Comparing
how much the four added *macro* variables took against the three added *micro* ones is therefore
a direct test of whether each extension opened a new information dimension or duplicated one
already present.

In [ ]:
# --- SHAP share displacement: baseline (11a) -> extended (11b) -----------------------
# Both specs are read at variant 'baseline_t1' over test years 2010-12, which they share,
# so the comparison carries no window confound: the same fitted-model machinery, the same
# evaluation years, only the feature set differs.
#
# NORMALISATION MATTERS HERE. Shares are computed WITHIN each model (summing to 1) before
# being averaged across models - the same convention as pooled_shares() above. Pooling the
# raw rows instead lets logistic regression dominate, because its raw |SHAP| magnitudes run
# an order of magnitude above the trees'. That is not a cosmetic difference: pooled rows put
# the added-macro share at 22.7% and show the micro block *rising* 8.3 -> 9.1%, whereas the
# within-model convention gives 24.3% and shows the micro block essentially flat (9.7 -> 9.4%,
# no consistent sign across models). Only the within-model figure is consistent with Figure 16
# and the Appendix H ranking, which use the same convention.

# The micro/macro split is declared per spec rather than via the shared MACRO_FEATURES set,
# because this cell needs to know which features are NEW in 11b relative to 11a, not merely
# which block each belongs to. (Both agree on membership: _is_micro above now also tests
# exclusion from MACRO_FEATURES, after a prefix-based version silently counted 'cr_impchg'
# as macro.)
MICRO_11A = {'r_cti', 'x_loans_gross', 'x_ib_liab', 'inc_nii', 'p_nim', 'inc_op', 'p_roa',
             'p_roe', 'x_dep', 'c_eqratio', 'l_assets', 'l_loans', 'r_loans_assets',
             'r_dep_liab', 'c_leverage'}
MICRO_11B = (MICRO_11A - {'l_loans'}) | {'cr_impchg', 'r_loans_dep', 'l_obsi'}


def _within_model_shares(spec):
    """|SHAP| share per feature per model, integrated config, baseline_t1. Sums to 100 per model."""
    df = pd.read_parquet(SHAP_FILES[spec])
    df = df[(df['variant'] == 'baseline_t1') & (df['feature_set'] == 'integrated')]
    cols = [c for c in df.columns if c.startswith('shap_')]
    pm = df.groupby('model_name')[cols].apply(lambda g: g.abs().mean())
    pm = pm.div(pm.sum(axis=1), axis=0) * 100
    pm.columns = [c[5:] for c in pm.columns]
    return pm


sh_a, sh_b = _within_model_shares('11a'), _within_model_shares('11b')
added_micro = sorted(MICRO_11B - MICRO_11A)
added_macro = sorted(set(sh_b.columns) - set(sh_a.columns) - set(added_micro))
dropped = sorted(set(sh_a.columns) - set(sh_b.columns))
print(f"added macro : {added_macro}\nadded micro : {added_micro}\ndropped     : {dropped}\n")

# Per-model view. Reported rather than only the average because the pooled mean cannot show
# whether the asymmetry is universal or driven by a few models - and the claim in the
# Discussion is the stronger, universal one.
rows = []
for m in MODEL_ORDER:
    mb_a = sh_a.loc[m, [c for c in sh_a.columns if c in MICRO_11A]].sum()
    mb_b = sh_b.loc[m, [c for c in sh_b.columns if c in MICRO_11B]].sum()
    rows.append({'Model': MODEL_LABELS_SHORT[m],
                 'Added macro (%)': sh_b.loc[m, added_macro].sum(),
                 'Added micro (%)': sh_b.loc[m, added_micro].sum(),
                 'Micro block 11a (%)': mb_a,
                 'Micro block 11b (%)': mb_b,
                 'Micro block change (pp)': mb_b - mb_a})
displacement = pd.DataFrame(rows).set_index('Model').round(2)
displacement.loc['MEAN'] = displacement.mean().round(2)
display(displacement)

# Per-feature detail: share and rank within the 30-feature extended set.
detail = pd.DataFrame({
    'Feature': [FI_NAMES.get(f, f) for f in added_macro + added_micro],
    'Code': added_macro + added_micro,
    'Level': ['Macro'] * len(added_macro) + ['Micro'] * len(added_micro),
    'Share (%)': [sh_b[f].mean() for f in added_macro + added_micro],
}).sort_values('Share (%)', ascending=False)
mean_share = sh_b.mean().sort_values(ascending=False)
detail['Rank (of %d)' % sh_b.shape[1]] = [
    list(mean_share.index).index(c) + 1 for c in detail['Code']]
detail['Share (%)'] = detail['Share (%)'].round(2)
display(detail.reset_index(drop=True))

_ma, _mi = displacement.loc['MEAN', 'Added macro (%)'], displacement.loc['MEAN', 'Added micro (%)']
print(f"\nAdded macro {_ma:.1f}% vs added micro {_mi:.1f}% of total attribution "
      f"({_ma/_mi:.0f}x).")
print(f"Added-macro share by model ranges {displacement['Added macro (%)'][:-1].min():.1f}"
      f"-{displacement['Added macro (%)'][:-1].max():.1f}%; added-micro "
      f"{displacement['Added micro (%)'][:-1].min():.1f}"
      f"-{displacement['Added micro (%)'][:-1].max():.1f}%.")
print("Micro block change is not consistently signed across models "
      f"({(displacement['Micro block change (pp)'][:-1] > 0).sum()} up, "
      f"{(displacement['Micro block change (pp)'][:-1] < 0).sum()} down) "
      "-> report as essentially unchanged, not as a rise.")

with pd.ExcelWriter(OUT / 'shap_displacement_11a_11b.xlsx', engine='openpyxl') as writer:
    displacement.to_excel(writer, sheet_name='Per model')
    detail.reset_index(drop=True).to_excel(writer, sheet_name='Added features', index=False)
print(f"\nSaved -> {OUT / 'shap_displacement_11a_11b.xlsx'}")

### ALE — functional form of the credit-to-GDP gap (Discussion)

Replots the saved ALE values (`outputs_*/ale_values_*.parquet`) for the credit-to-GDP gap in
the integrated config, styled to match the thesis. Shows the **lag-1** threshold (sharp at
gap ≈ 0) against **lag-2** (washed out) — the horizon-sensitivity contrast. Exports to
`results_discussion/`.

In [ ]:
# Accumulated Local Effects (ALE) — credit-to-GDP gap, integrated config.
# Replotted from the saved ALE tables so styling matches the thesis (one clean curve per model).
import numpy as np

ALE_BASE = BASE / 'outputs/attribution'
OUTPUT_DISC = BASE / 'outputs/figures' / 'results_discussion'   # same folder as the cross-spec figs
OUTPUT_DISC.mkdir(parents=True, exist_ok=True)

# ALE shows one line per MODEL (not per feature), so it needs a per-model colour map.
MODEL_ORDER  = ['logistic_regression', 'decision_tree', 'random_forest', 'xgboost', 'lightgbm', 'mlp']
MODEL_LABELS_SHORT = {'logistic_regression': 'LR', 'decision_tree': 'DT', 'random_forest': 'RF',
                'xgboost': 'XGBoost', 'lightgbm': 'LightGBM', 'mlp': 'MLP'}
MODEL_COLORS = {'logistic_regression': '#2ca02c', 'decision_tree': '#1f77b4', 'random_forest': '#9467bd',
                'xgboost': '#8c564b', 'lightgbm': '#ff7f0e', 'mlp': '#d62728'}

def ale_mean_curves(spec, feature, variant='baseline_t1', n_grid=60, trim=(0.02, 0.98)):
    """Return (grid, {model: mean ALE}) for one feature in the integrated config.
    The ALE table is long: each row is one grid point of one feature, and only that feature's
    column holds the x-value (the rest are NaN), with `eff` the ALE effect. The grid is
    quantile-based and differs slightly per test year, so each year's curve is interpolated
    onto a shared grid before averaging — this removes the year-overlap zigzag seen in the
    original per-notebook figures and yields one representative curve per model."""
    df = pd.read_parquet(ALE_BASE / f'outputs_{spec}' / f'ale_values_{spec}.parquet')
    sub = df[(df['variant'] == variant) & df[feature].notna()]
    lo, hi = sub[feature].quantile(trim)            # drop extreme grid points that stretch the axis
    grid = np.linspace(lo, hi, n_grid)
    curves = {}
    for m, md in sub.groupby('model_name'):
        per_year = []
        for _, yd in md.groupby('test_year'):
            yd = yd.sort_values(feature)            # np.interp requires ascending x
            per_year.append(np.interp(grid, yd[feature], yd['eff']))
        curves[m] = np.mean(per_year, axis=0)       # mean ALE across the test years
    return grid, curves

def plot_ale(spec, feature, xlabel, lag_label, fname, threshold0=True):
    grid, curves = ale_mean_curves(spec, feature)
    fig, ax = plt.subplots(figsize=(8, 5))
    for m in MODEL_ORDER:
        if m in curves:
            ax.plot(grid, curves[m], color=MODEL_COLORS[m], lw=1.8, label=MODEL_LABELS_SHORT[m])
    ax.axhline(0, ls=':', color='black', lw=0.8)        # no-effect reference
    if threshold0:
        ax.axvline(0, ls='--', color='grey', lw=1.0)    # credit-gap threshold (Borio/Drehmann EWI)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('ALE effect on crisis probability', fontsize=10)
    #ax.set_title(f'ALE — {xlabel} ({lag_label}, integrated config)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, ncol=2)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DISC / fname, bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved -->', fname)

# Credit-to-GDP gap enters with the extended set: lag-1 shows a sharp threshold at gap=0,
# lag-2 is flat for most models — the horizon-sensitivity contrast.
plot_ale('11b', 'credit_gap', 'Credit-to-GDP gap', 'lag-1', 'ale_credit_gap_lag1.png')
plot_ale('11d', 'credit_gap', 'Credit-to-GDP gap', 'lag-2', 'ale_credit_gap_lag2.png')

In [ ]:
# Accumulated Local Effects (ALE) — credit-to-GDP gap, integrated config.
# Replotted from the saved ALE tables so styling matches the thesis (one clean curve per model).
import numpy as np

ALE_BASE = BASE / 'outputs/attribution'
OUTPUT_DISC = BASE / 'outputs/figures' / 'results_discussion'   # same folder as the cross-spec figs
OUTPUT_DISC.mkdir(parents=True, exist_ok=True)

# ALE shows one line per MODEL (not per feature), so it needs a per-model colour map.
MODEL_ORDER  = ['logistic_regression', 'decision_tree', 'random_forest', 'xgboost', 'lightgbm', 'mlp']
MODEL_LABELS_SHORT = {'logistic_regression': 'LR', 'decision_tree': 'DT', 'random_forest': 'RF',
                'xgboost': 'XGBoost', 'lightgbm': 'LightGBM', 'mlp': 'MLP'}
MODEL_COLORS = {'logistic_regression': '#2ca02c', 'decision_tree': '#1f77b4', 'random_forest': '#9467bd',
                'xgboost': '#8c564b', 'lightgbm': '#ff7f0e', 'mlp': '#d62728'}

def ale_mean_curves(spec, feature, variant='baseline_t1', n_grid=60, trim=(0.02, 0.98)):
    """Return (grid, {model: mean ALE}) for one feature in the integrated config.
    The ALE table is long: each row is one grid point of one feature, and only that feature's
    column holds the x-value (the rest are NaN), with `eff` the ALE effect. The grid is
    quantile-based and differs slightly per test year, so each year's curve is interpolated
    onto a shared grid before averaging — this removes the year-overlap zigzag seen in the
    original per-notebook figures and yields one representative curve per model."""
    df = pd.read_parquet(ALE_BASE / f'outputs_{spec}' / f'ale_values_{spec}.parquet')
    sub = df[(df['variant'] == variant) & df[feature].notna()]
    lo, hi = sub[feature].quantile(trim)            # drop extreme grid points that stretch the axis
    grid = np.linspace(lo, hi, n_grid)
    curves = {}
    for m, md in sub.groupby('model_name'):
        per_year = []
        for _, yd in md.groupby('test_year'):
            yd = yd.sort_values(feature)            # np.interp requires ascending x
            per_year.append(np.interp(grid, yd[feature], yd['eff']))
        curves[m] = np.mean(per_year, axis=0)       # mean ALE across the test years
    return grid, curves

def plot_ale(spec, feature, xlabel, lag_label, fname, threshold0=True):
    grid, curves = ale_mean_curves(spec, feature)
    fig, ax = plt.subplots(figsize=(8, 5))
    for m in MODEL_ORDER:
        if m in curves:
            ax.plot(grid, curves[m], color=MODEL_COLORS[m], lw=1.8, label=MODEL_LABELS_SHORT[m])
    ax.axhline(0, ls=':', color='black', lw=0.8)        # no-effect reference
    if threshold0:
        ax.axvline(0, ls='--', color='grey', lw=1.0)    # credit-gap threshold (Borio/Drehmann EWI)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('ALE effect on crisis probability', fontsize=10)
    #ax.set_title(f'ALE — {xlabel} ({lag_label}, integrated config)', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, ncol=2)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DISC / fname, bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved -->', fname)

# Credit-to-GDP gap enters with the extended set: lag-1 shows a sharp threshold at gap=0,
# lag-2 is flat for most models — the horizon-sensitivity contrast.
plot_ale('11b', 'NGDP_RPCH', 'Real GDP growth', 'lag-1', 'ale_NGDP_RPCH_lag1.png')
plot_ale('11d', 'NGDP_RPCH', 'Real GDP growth', 'lag-2', 'ale_NGDP_RPCH_lag2.png')

## Combined panels for pasting (Discussion)
Single-figure versions: 2x3 cross-spec bump and 2x2 ALE (two macro features x two horizons).

In [ ]:
# Combined cross-spec bump - single 2x3 figure (rows: lag-1, lag-2; cols: model class).
# Same per-panel logic as build_bump, drawn into a shared grid so it can be pasted as one figure.
def build_bump_combined(fname='fi_crossspec_bump_combined.png'):
    rows = [('11a', '11b', 'lag-1'), ('11c', '11d', 'lag-2')]
    fig, axes = plt.subplots(2, 3, figsize=(21, 13.5), gridspec_kw={'wspace': 1.25, 'hspace': 0.3})
    for r, (spec_b, spec_e, lag_label) in enumerate(rows):
        base, ext = pooled_shares(spec_b), pooled_shares(spec_e)
        base_feats = set(base['feat'])
        for c, cls in enumerate(CLASS_ORDER):
            ax = axes[r][c]
            rb = base[base['class'] == cls].sort_values('share', ascending=False).reset_index(drop=True)
            re_ = ext[ext['class'] == cls].sort_values('share', ascending=False).reset_index(drop=True)
            topb = {row['feat']: i + 1 for i, row in rb.head(N_SHOW).iterrows()}
            tope = {row['feat']: i + 1 for i, row in re_.head(N_SHOW).iterrows()}
            for feat in set(topb) | set(tope):
                col, lbl = _feat_color(feat), BUMP_SHORT.get(feat, FI_NAMES.get(feat, feat))
                if feat in topb and feat in tope:
                    ax.plot([0, 1.6], [topb[feat], tope[feat]], '-o', color=col, lw=2, ms=6, alpha=0.85)
                    ax.text(-0.13, topb[feat], lbl, ha='right', va='center', fontsize=6.5, color=col)
                    ax.text(1.73, tope[feat], lbl, ha='left', va='center', fontsize=6.5, color=col)
                elif feat in tope:
                    tag = ' (new)' if feat not in base_feats else ' (↑)'
                    ax.plot([1.6], [tope[feat]], 'o', color=col, ms=8)
                    ax.text(1.73, tope[feat], lbl + tag, ha='left', va='center', fontsize=6.5,
                            color=col, fontweight='bold' if tag == ' (new)' else 'normal')
                else:
                    ax.plot([0], [topb[feat]], 'o', color=col, ms=6, alpha=0.45)
                    ax.text(-0.13, topb[feat], lbl + ' (↓)', ha='right', va='center',
                            fontsize=6.5, color=col, alpha=0.6)
            ax.set_title(f'{cls} ({lag_label})', fontsize=11, fontweight='bold')
            ax.set_xlim(-2.0, 3.4); ax.set_xticks([0, 1.6])
            ax.set_xticklabels(['Baseline', 'Extended'], fontsize=9)
            ax.set_ylim(N_SHOW + 0.5, 0.5); ax.set_yticks(range(1, N_SHOW + 1))
            if c == 0:
                ax.set_ylabel('Rank within integrated config', fontsize=6.5)
            for sp in ['top', 'right']:
                ax.spines[sp].set_visible(False)
    handles = [mpatches.Patch(color=MICRO_COLOR, label='Micro'),
               mpatches.Patch(color=MACRO_COLOR, label='Macro'),
               mpatches.Patch(color=CREDIT_COLOR, label='Credit-cycle (extended set)')]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.01))
    #fig.suptitle('Integrated feature-importance rank: Baseline → Extended set, pooled by model class',
    #             fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.savefig(OUTPUT_DISC / fname, bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved -->', fname)

build_bump_combined()

In [ ]:
# Combined ALE - one figure per macro feature, lag-1 stacked vertically above lag-2.
# Each feature gets its own figure; both horizons come from the extended set.
def build_ale_pair(feature, xlabel, fname):
    layout = [('11b', 'lag-1'), ('11d', 'lag-2')]            # rows: lag-1 on top, lag-2 beneath
    fig, axes = plt.subplots(2, 1, figsize=(7.5, 10))
    for r, (spec, lag_label) in enumerate(layout):
        ax = axes[r]
        grid, curves = ale_mean_curves(spec, feature)
        for m in MODEL_ORDER:
            if m in curves:
                ax.plot(grid, curves[m], color=MODEL_COLORS[m], lw=1.8, label=MODEL_LABELS_SHORT[m])
        ax.axhline(0, ls=':', color='black', lw=0.8)     # no-effect reference
        ax.axvline(0, ls='--', color='grey', lw=1.0)     # zero reference (credit-gap EWI / zero growth)
        ax.set_title(lag_label, fontsize=11, fontweight='bold')
        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_ylabel('ALE effect on crisis probability', fontsize=9)
        for sp in ['top', 'right']:
            ax.spines[sp].set_visible(False)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=6, fontsize=9, bbox_to_anchor=(0.5, -0.01))
    #fig.suptitle(f'ALE - {xlabel}, integrated config (lag-1 vs lag-2)', fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0.04, 1, 0.96])
    plt.savefig(OUTPUT_DISC / fname, bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved -->', fname)

build_ale_pair('NGDP_RPCH', 'Real GDP growth', 'ale_real_gdp_growth_pair.png')
build_ale_pair('credit_gap', 'Credit-to-GDP gap', 'ale_credit_gap_pair.png')

## RT1 robustness bump (Discussion)
Full sample → crisis years excluded, integrated config, lag-1, windows matched to RT1's 2011-2012.

In [ ]:
# RT1 robustness bump - integrated feature-importance rank, full sample -> crisis years excluded.
# Rows: indicator set (baseline 11a / extended 11b) at lag-1.  Cols: model class.
# Windows restricted to 2011-2012 (RT1's only two windows) so the shift is not confounded by 2010,
# which has no RT1 counterpart. RT1 never adds features, so a feature entering the top-10 is a rise (up).
def pooled_shares_v(spec, variant, years=None):
    """pooled_shares generalised to any variant + an optional test_year filter (for window matching)."""
    df = pd.read_parquet(SHAP_FILES[spec])
    df = df[(df['variant'] == variant) & (df['feature_set'] == 'integrated')]
    if years is not None:
        df = df[df['test_year'].isin(years)]            # match windows across variants
    cols = [c for c in df.columns if c.startswith('shap_')]
    rows = []
    for m, sub in df.groupby('model_name'):
        imp = sub[cols].abs().mean()
        imp = imp[imp > 0]
        share = imp / imp.sum()                          # within-model share before pooling by class
        for c, s in share.items():
            rows.append({'class': CLASS[m], 'feat': c[5:], 'share': s})
    return pd.DataFrame(rows).groupby(['class', 'feat'])['share'].mean().reset_index()

def build_rt1_bump(fname='fi_rt1_bump_lag1.png'):
    rows = [('11a', 'baseline set'), ('11b', 'extended set')]
    OVERLAP = [2011, 2012]                               # intersection of baseline_t1 and rt1 windows
    fig, axes = plt.subplots(2, 3, figsize=(21, 13.5), gridspec_kw={'wspace': 1.25, 'hspace': 0.3})
    for r, (spec, set_label) in enumerate(rows):
        base = pooled_shares_v(spec, 'baseline_t1', years=OVERLAP)
        rt1  = pooled_shares_v(spec, 'rt1', years=OVERLAP)
        for c, cls in enumerate(CLASS_ORDER):
            ax = axes[r][c]
            rb = base[base['class'] == cls].sort_values('share', ascending=False).reset_index(drop=True)
            rr = rt1[rt1['class'] == cls].sort_values('share', ascending=False).reset_index(drop=True)
            topb = {row['feat']: i + 1 for i, row in rb.head(N_SHOW).iterrows()}
            topr = {row['feat']: i + 1 for i, row in rr.head(N_SHOW).iterrows()}
            for feat in set(topb) | set(topr):
                col, lbl = _feat_color(feat), BUMP_SHORT.get(feat, FI_NAMES.get(feat, feat))
                if feat in topb and feat in topr:
                    ax.plot([0, 1.6], [topb[feat], topr[feat]], '-o', color=col, lw=2, ms=6, alpha=0.85)
                    ax.text(-0.13, topb[feat], lbl, ha='right', va='center', fontsize=6.5, color=col)
                    ax.text(1.73, topr[feat], lbl, ha='left', va='center', fontsize=6.5, color=col)
                elif feat in topr:
                    ax.plot([1.6], [topr[feat]], 'o', color=col, ms=8)
                    ax.text(1.73, topr[feat], lbl + ' (↑)', ha='left', va='center', fontsize=6.5,
                            color=col, fontweight='bold')
                else:
                    ax.plot([0], [topb[feat]], 'o', color=col, ms=6, alpha=0.45)
                    ax.text(-0.13, topb[feat], lbl + ' (↓)', ha='right', va='center',
                            fontsize=6.5, color=col, alpha=0.6)
            ax.set_title(f'{cls} ({set_label})', fontsize=11, fontweight='bold')
            ax.set_xlim(-2.0, 3.4); ax.set_xticks([0, 1.6])
            ax.set_xticklabels(['Full sample', 'Crisis yrs excl.'], fontsize=9)
            ax.set_ylim(N_SHOW + 0.5, 0.5); ax.set_yticks(range(1, N_SHOW + 1))
            if c == 0:
                ax.set_ylabel('Rank within integrated config', fontsize=6.5)
            for sp in ['top', 'right']:
                ax.spines[sp].set_visible(False)
    handles = [mpatches.Patch(color=MICRO_COLOR, label='Micro'),
               mpatches.Patch(color=MACRO_COLOR, label='Macro'),
               mpatches.Patch(color=CREDIT_COLOR, label='Credit-cycle (extended set)')]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.01))
    #fig.suptitle('Integrated feature-importance rank: full sample → crisis years excluded (lag-1)',
    #             #'excluded (lag-1), matched 2011-2012 windows, pooled by model class',
    #             fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.savefig(OUTPUT_DISC / fname, bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved -->', fname)

build_rt1_bump()

## Crisis-definition robustness — Reinhart & Rogoff (2011) vs Laeven & Valencia

Results figure + **Appendix I** (Tables I1–I7).

Both arms run on an **identical matched sample** (2,157 bank-years, 13 countries, same
features, expanding-window design and tuning budget) with only the crisis **label** swapped,
so differences are attributable solely to the crisis definition.

⚠️ Both arms are restricted to the **shared test windows 2010–2012**. R&R natively evaluates
2010–2014 because its broader rule keeps countries labelled in crisis through 2013–14;
averaging each arm over its own years would confound the definition effect with which years
each side covers. For the same reason `load_spec_metrics()` and the `h3_auroc_summary` sheet
are **not** reused here — both aggregate over whatever windows a file happens to contain.

⚠️ The figure reports **AUROC only**: AUPRC's no-skill baseline is the positive prevalence,
which is 26.4% under L&V against 70.9% under R&R on these same rows, so an AUPRC delta would
be a base-rate artefact. AUPRC is quoted within each definition only.

In [ ]:
# CRISIS-DEFINITION ROBUSTNESS — Reinhart & Rogoff (2011) vs Laeven & Valencia
# Figure (Results) + Appendix I Tables I3-I5
# Both arms were estimated on an IDENTICAL matched sample (2,157 bank-years, 13
# countries, same features / expanding-window design / tuning budget) with only the
# crisis LABEL swapped, so any difference here is attributable solely to the crisis
# definition. That is also why the comparison is R&R vs L&V-ISOLATED and never vs the
# main 25-country results — those differ in sample as well as definition.
#
# Three constraints are enforced below, each one a defect this project has hit before:
#
# 1. MATCHED TEST WINDOWS. R&R evaluates 2010-2014 (its broader rule keeps countries
#    labelled in crisis through 2013-14); L&V-isolated only 2010-2012. Averaging each
#    arm over its own years would place two crisis-heavy years on the R&R side alone —
#    the same unmatched-window defect that produced three wrong claims in D7. Note that
#    load_spec_metrics() CANNOT be reused here: it averages over whatever years a file
#    contains, which is exactly the wrong behaviour for this comparison.
# 2. AUROC ONLY in the figure. AUPRC's no-skill baseline IS the positive prevalence,
#    which is 26.4% under L&V against 70.9% under R&R on these same rows. An AUPRC
#    delta would be almost entirely a base-rate artefact rather than a change in
#    precision-recall quality. AUPRC is reported within each definition in the table
#    notes instead, never across them.
# 3. lr_no_dummies is dropped — a diagnostic variant, not one of the six models (the
#    same exclusion load_spec_metrics applies).

import matplotlib.patches as mpatches

RR_DIR = BASE.parent / 'Different Definition' / 'Code_RR'
RR_OVERLAP = [2010, 2011, 2012]          # the three windows both definitions share
# L&V first so the incumbent definition leads every table, matching the drafted appendix.
# Order matters only for column layout — rr_delta is always computed as R&R minus L&V.
RR_ARMS = {'LViso': 'L&V', 'RR': 'R&R'}  # folder suffix -> short label used in tables


def rr_arm_metrics(spec, arm):
    """Mean AUROC/AUPRC per model x config for one arm, on the shared test windows.

    `spec` is '11a' (baseline set) or '11b' (extended set); `arm` is 'RR' or 'LViso'.
    The assert is deliberate: if either arm ever loses a window the comparison silently
    stops being matched, which is the failure mode this whole cell is guarding against.
    """
    f = RR_DIR / f'results_{spec}_{arm}' / f'results_{spec}_{arm}_final.xlsx'
    d = pd.read_excel(f, sheet_name='all_results')
    d = d[(d['variant'] == 'baseline_t1') &
          (d['model'] != 'lr_no_dummies') &
          (d['test_year'].isin(RR_OVERLAP))]
    n_win = d.groupby(['model', 'dataset'])['test_year'].nunique()
    assert set(n_win) == {len(RR_OVERLAP)}, \
        f'{spec}_{arm}: unmatched windows {sorted(set(n_win))} — comparison invalid'
    return d.groupby(['model', 'dataset'])[['auroc', 'auprc']].mean()


RR_SPECS = [('11a', 'baseline indicator set'), ('11b', 'extended indicator set')]
rr_metrics, rr_delta = {}, {}
for _spec, _ in RR_SPECS:
    rr_metrics[_spec] = {a: rr_arm_metrics(_spec, a) for a in RR_ARMS}
    rr_delta[_spec] = rr_metrics[_spec]['RR']['auroc'] - rr_metrics[_spec]['LViso']['auroc']


# --- FIGURE: AUROC delta (R&R - L&V) per model and configuration ---
# Deliberately built to the same grammar as Figure 2 (nb11b_combined_delta): 3x1 vertical
# panels, one per CONFIGURATION, six models on the x-axis, two alpha-differentiated bars per
# model sharing the configuration colour. Only the meaning of the two bars changes: Figure 2
# contrasts AUROC against AUPRC, whereas here both bars are AUROC and the contrast is the
# BASELINE against the EXTENDED indicator set.
#
# AUPRC cannot take the second slot in this figure. Its no-skill baseline IS the positive
# prevalence, which on these test windows is 85.6% under R&R against 21.2% under L&V, so an
# AUPRC delta across definitions would be a base-rate artefact rather than a change in
# precision-recall quality. AUPRC is reported within each definition in the table notes only.
from matplotlib.legend_handler import HandlerTuple

fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharey=False)
width = 0.35
x = np.arange(len(MODEL_ORDER))

for ax, config in zip(axes, DATASET_ORDER):
    color = DATASET_COLORS[config]
    vals_base = [rr_delta['11a'].loc[(mdl, config)] for mdl in MODEL_ORDER]
    vals_ext = [rr_delta['11b'].loc[(mdl, config)] for mdl in MODEL_ORDER]

    bars1 = ax.bar(x - width / 2, vals_base, width, color=color, alpha=0.85,
                   edgecolor='white', linewidth=0.5)
    bars2 = ax.bar(x + width / 2, vals_ext, width, color=color, alpha=0.4,
                   edgecolor='white', linewidth=0.5)

    ax.axhline(0, color='black', lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS_SHORT[mdl] for mdl in MODEL_ORDER],
                       rotation=25, ha='right', fontsize=9)
    ax.set_title(DATASET_LABELS[config], fontsize=10, fontweight='bold')
    ax.set_ylabel('Delta  (R&R − L&V)', fontsize=9)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    ax.margins(y=0.2)

    for bar, v in zip([*bars1, *bars2], [*vals_base, *vals_ext]):
        if not np.isnan(v) and abs(v) >= 0.005:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    v + (0.003 if v >= 0 else -0.003),
                    f'{v:+.3f}', ha='center',
                    va='bottom' if v >= 0 else 'top', fontsize=7.5)

patches_base = tuple(mpatches.Patch(color=DATASET_COLORS[c], alpha=0.85)
                     for c in DATASET_ORDER)
patches_ext = tuple(mpatches.Patch(color=DATASET_COLORS[c], alpha=0.4)
                    for c in DATASET_ORDER)
fig.legend(
    handles=[patches_base, patches_ext],
    labels=['Baseline indicator set', 'Extended indicator set'],
    handler_map={tuple: HandlerTuple(ndivide=None, pad=0.1)},
    fontsize=9, loc='lower center', ncol=2, bbox_to_anchor=(0.5, 0.01)
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(OUTPUT / 'rr_vs_lv_definition_delta.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved → rr_vs_lv_definition_delta.png (per model and configuration)')

rr_pooled = {(spec, ds): np.array([rr_delta[spec].loc[(mdl, ds)] for mdl in MODEL_ORDER])
             for spec, _ in RR_SPECS for ds in DATASET_ORDER}

# --- figure note: the dispersion the pooled bars average away ---
# These counts belong in the figure note so a pooled mean is never read as "every model".
_neg_total = 0
for spec, label in RR_SPECS:
    _neg_panel = sum(int((rr_pooled[(spec, ds)] < 0).sum()) for ds in DATASET_ORDER)
    _neg_total += _neg_panel
    print(f'  {label}: {_neg_panel}/18 model × configuration deltas negative')
    for ds in DATASET_ORDER:
        v = rr_pooled[(spec, ds)]
        print(f'    {ds:11s} mean {v.mean():+.3f} | range {v.min():+.3f} to {v.max():+.3f} '
              f'| sd {v.std(ddof=1):.3f} | negative {(v < 0).sum()}/6')
print(f'  TOTAL: {_neg_total}/36 negative; test windows {RR_OVERLAP} (n = {len(RR_OVERLAP)})')


# --- TABLES I3 / I4: AUROC by model, configuration and definition ---
def rr_auroc_table(spec):
    """Wide publication table: six L&V/R&R AUROC columns plus the three deltas."""
    cols = {}
    for arm, lab in RR_ARMS.items():
        for ds in DATASET_ORDER:
            cols[f'{lab} {DATASET_LABELS[ds].replace("-only", "").lower()}'] = \
                rr_metrics[spec][arm]['auroc'].xs(ds, level='dataset')
    out = pd.DataFrame(cols)
    for ds in DATASET_ORDER:
        out[f'Δ {DATASET_LABELS[ds].replace("-only", "").lower()}'] = \
            rr_delta[spec].xs(ds, level='dataset')
    out.index = [MODEL_LABELS_SHORT[m] for m in out.index]
    return out.reindex([MODEL_LABELS_SHORT[m] for m in MODEL_ORDER]).round(3)


table_i3 = rr_auroc_table('11a')
table_i4 = rr_auroc_table('11b')

# --- TABLE I5: what does extending the indicator set do under each definition? ---
# Tests whether the main-results extension finding is a property of the extension or of the
# L&V labelling. Covers ALL THREE configurations, not micro alone: restricted to micro it
# would show only that micro's degradation is L&V-specific and would hide the more
# informative result — that under L&V the extension REALLOCATES (micro −0.048, macro +0.033,
# integrated +0.022) whereas under R&R it is mildly negative across the board with no
# reallocation and integrated hurt most (−0.037). What is definition-specific is therefore
# the whole asymmetry, not just the micro leg. Showing all three also keeps I5 consistent
# with I3 and I4, which both report every configuration.
table_i5 = pd.DataFrame({
    f'{lab} {DATASET_LABELS[ds].replace("-only", "").lower()}':
        (rr_metrics['11b'][arm]['auroc'].xs(ds, level='dataset')
         - rr_metrics['11a'][arm]['auroc'].xs(ds, level='dataset'))
    for ds in DATASET_ORDER for arm, lab in RR_ARMS.items()
})
table_i5.index = [MODEL_LABELS_SHORT[m] for m in table_i5.index]
table_i5 = table_i5.reindex([MODEL_LABELS_SHORT[m] for m in MODEL_ORDER]).round(3)

# AUPRC ranges — quoted in the table notes, WITHIN each definition only (see constraint 2)
rr_auprc_ranges = {
    spec: {lab: (rr_metrics[spec][arm]['auprc'].min(), rr_metrics[spec][arm]['auprc'].max())
           for arm, lab in RR_ARMS.items()}
    for spec, _ in RR_SPECS
}

print('\n=== Table I3 — AUROC, baseline indicator set ===')
print(table_i3.to_string())
print('\n=== Table I4 — AUROC, extended indicator set ===')
print(table_i4.to_string())
print('\n=== Table I5 — AUROC change from extending the indicator set, by definition ===')
print(table_i5.to_string())
for spec, _ in RR_SPECS:
    print(f'\nAUPRC range {spec} (within-definition only): ' +
          ' | '.join(f'{lab} {lo:.3f}-{hi:.3f}'
                     for lab, (lo, hi) in rr_auprc_ranges[spec].items()))

# Sign summary used for the Results sentence
print('\n=== how many of 6 models improve under R&R ===')
for spec, label in RR_SPECS:
    for ds in DATASET_ORDER:
        v = rr_delta[spec].xs(ds, level='dataset').reindex(MODEL_ORDER).values
        print(f'{spec} {ds:11s}: {(v > 0).sum()}/6 positive, mean {v.mean():+.3f}, '
              f'range {v.min():+.3f} to {v.max():+.3f}')

In [ ]:
# APPENDIX I — Tables I1, I2, I6, I7 + export of the full appendix
# Runs AFTER the figure cell above (it reuses rr_metrics / RR_DIR / RR_OVERLAP).
# Previously these tables were produced by a throwaway scratchpad script, which meant
# Appendix I failed the third citability condition (reproducible from a notebook in the
# repo). This cell closes that gap — every number in Appendix I is now rebuilt here.

# --- TABLES I1 / I2: how the two dating rules differ, before any modelling ---
# Computed from the crisis panels rather than the modelling panels, because the
# label-correspondence claim is about the DEFINITIONS, not about the estimation sample.
lv_panel = pd.read_parquet(
    BASE / 'data/crisis' / 'clean' / 'parquet_saves' / 'crisis_panel.parquet')
rr_panel = pd.read_parquet(RR_DIR / 'crisis_panel_RR.parquet')

# Restrict to what the two databases actually share: the 13 usable R&R countries and the
# overlapping years. Ireland and 2015-16 are already absent from the R&R panel (entirely
# missing in the source file), so the inner merge does the filtering.
corr = rr_panel.merge(lv_panel, on=['country_iso3', 'year'],
                      suffixes=('_rr', '_lv'), how='inner').dropna()
table_i1 = pd.crosstab(corr['crisis_rr'], corr['crisis_lv'])
table_i1.index = ['R&R = 0', 'R&R = 1']
table_i1.columns = ['L&V = 0', 'L&V = 1']
i1_agreement = (corr['crisis_rr'] == corr['crisis_lv']).mean()
i1_lv_only = corr[(corr['crisis_lv'] == 1) & (corr['crisis_rr'] == 0)]

# Table I2: crisis COUNTRY-years per definition. Collapsed from the matched modelling
# sample so the counts are not inflated by how many banks each country contributes.
me_rr = pd.read_parquet(RR_DIR / 'me_merged_base_RR.parquet')
me_lv = pd.read_parquet(RR_DIR / 'me_merged_base_LViso.parquet')
i2_years = [2008, 2009, 2010, 2011, 2012, 2013, 2014]
table_i2 = pd.DataFrame({
    lab: (df[df['crisis'] == 1].groupby('year')['country_iso'].nunique()
          .reindex(i2_years).fillna(0).astype(int))
    for lab, df in [('R&R (2011)', me_rr), ('L&V (2026)', me_lv)]
}).T
# Prevalence on the matched bank-year sample — the figure quoted in §3.3.3 and §4.2
i2_prevalence = {'R&R': me_rr['crisis'].mean(), 'L&V': me_lv['crisis'].mean(),
                 'n_bank_years': len(me_rr), 'n_countries': me_rr['country_iso'].nunique()}

# --- TABLE I6: H3 interaction terms under each definition ---
# The H3 test is the integrated LOGISTIC REGRESSION with vs without interaction terms.
# It reads h3_RAW and re-averages over RR_OVERLAP, NOT the h3_auroc_summary sheet:
# that sheet is pre-aggregated over every test year in the file, so for the R&R arms it
# is a five-window mean (integrated 0.901) against the L&V arms' three (0.743). Using it
# would reintroduce the unmatched-window confound at the aggregate level and would also
# understate the smooth-interaction delta as +0.004, contradicting the +0.006 bound
# quoted in the Results text.
i6_rows = []
for spec, set_label in [('11a', 'Baseline (lag-1)'), ('11b', 'Extended (lag-1)')]:
    for arm, lab in RR_ARMS.items():
        h3_raw = pd.read_excel(
            RR_DIR / f'results_{spec}_{arm}' / f'results_{spec}_{arm}_final.xlsx',
            sheet_name='h3_raw')
        h3 = (h3_raw[h3_raw['test_year'].isin(RR_OVERLAP)]
              .groupby('dataset')['auroc'].mean())
        base = h3.get('integrated', np.nan)
        i6_rows.append({
            'Indicator set': set_label, 'Definition': lab, 'Integrated': base,
            '+ smooth interactions': h3.get('integrated_h3', np.nan),
            'Δ smooth': h3.get('integrated_h3', np.nan) - base,
            '+ threshold interactions': h3.get('integrated_h3_thr', np.nan),
            'Δ threshold': h3.get('integrated_h3_thr', np.nan) - base,
        })
table_i6 = pd.DataFrame(i6_rows).round(3)

# --- TABLE I7: DeLong counts, WINDOW-level ---
# NB: this is a window-level count (6 models x 3 windows = 18 tests per row), NOT the
# model-level majority rule used for the asterisks in the main results tables. The two
# bases give different numbers and must never be mixed in one sentence.
i7_rows = []
for spec, set_label in [('11a', 'Baseline'), ('11b', 'Extended')]:
    for arm, lab in RR_ARMS.items():
        raw = pd.read_excel(RR_DIR / f'results_{spec}_{arm}' / f'results_{spec}_{arm}_final.xlsx',
                            sheet_name='delong_raw')
        raw = raw[(raw['variant'] == 'baseline_t1') &
                  (raw['test_year'].isin(RR_OVERLAP)) &
                  (raw['model'] != 'lr_no_dummies')]
        for comp, comp_label in [('integrated_vs_micro', 'vs micro'),
                                 ('integrated_vs_macro', 'vs macro')]:
            sub = raw[raw['comparison'] == comp]
            i7_rows.append({
                'Indicator set': set_label, 'Definition': lab, 'Comparison': comp_label,
                'Significant wins': int(((sub['sig_05']) & (sub['z'] > 0)).sum()),
                'Significant losses': int(((sub['sig_05']) & (sub['z'] < 0)).sum()),
                'n tests': len(sub),
            })
table_i7 = pd.DataFrame(i7_rows)

# --- EXPORT ---
_out = BASE / 'outputs/tables' / 'appendix_i_crisis_definition.xlsx'
with pd.ExcelWriter(_out, engine='openpyxl') as xw:
    table_i1.to_excel(xw, sheet_name='I1_label_correspondence')
    table_i2.to_excel(xw, sheet_name='I2_crisis_country_years')
    table_i3.to_excel(xw, sheet_name='I3_auroc_baseline')
    table_i4.to_excel(xw, sheet_name='I4_auroc_extended')
    table_i5.to_excel(xw, sheet_name='I5_micro_extension_effect')
    table_i6.to_excel(xw, sheet_name='I6_interactions', index=False)
    table_i7.to_excel(xw, sheet_name='I7_delong_counts', index=False)

print('=== Table I1 — label correspondence ===')
print(table_i1.to_string())
print(f'agreement {i1_agreement:.1%} ({int((corr["crisis_rr"] == corr["crisis_lv"]).sum())}/{len(corr)});'
      f' L&V-only country-years: '
      f'{", ".join(f"{r.country_iso3} {r.year}" for r in i1_lv_only.itertuples())}')
print('\n=== Table I2 — crisis country-years ===')
print(table_i2.to_string())
print(f'matched sample: {i2_prevalence["n_bank_years"]} bank-years, '
      f'{i2_prevalence["n_countries"]} countries; prevalence '
      f'R&R {i2_prevalence["R&R"]:.1%} vs L&V {i2_prevalence["L&V"]:.1%}')
print('\n=== Table I6 — interaction terms ===')
print(table_i6.to_string(index=False))
print('\n=== Table I7 — DeLong window-level counts ===')
print(table_i7.to_string(index=False))
print(f'\nSaved → {_out.name}')

## Appendix E2 — matched-window AUROC (comparisons only)

Table E1 reports each specification over its **own** test years, which is right for a
*level*. This section adds the companion table for *differences*: every specification
restricted to the shared window 2011-2012, so cross-column reads are like-for-like.
Also regenerates the lag-effect figure on matched windows.

In [ ]:
# Appendix E2 — matched-window AUROC + the matched lag-effect figure
# WHY THIS EXISTS
# Every column of Appendix E Table E1 is averaged over that specification's OWN
# test years. That is correct for reporting a LEVEL ("how did this spec do?"),
# because restricting a spec to fewer years would throw away a real evaluation
# year for nothing.
#
# It is NOT valid for a COMPARISON. Subtracting two numbers computed over
# different years does not give a difference between specifications, it gives a
# difference between specifications PLUS a difference between years. Here that
# second term is large: mean AUROC across models is 0.477 in 2010 against 0.764
# and 0.799 in 2011/2012, i.e. 2010 is at or below chance. The lag-1 specs are
# scored on it and the lag-2 specs are not.
#
# WHY 2010 IS MISSING FROM THE LAG-2 SPECS (checked, not assumed)
# It is not that a lag-2 forecast for 2010 is impossible: the lag-2 panel
# contains 76 crisis observations in 2010. Lagging by two years costs one extra
# year at the FRONT of the panel (lag-1 starts 2005, lag-2 starts 2006), so the
# same `min_train_years=5` warm-up rule lands one year later. The window gap is
# therefore an artefact of the warm-up rule, not a property of the horizon.
#
# WHAT THIS CELL BUILDS
#   1. `e2_auroc_matched`  — every specification restricted to 2011-2012, so any
#                            cross-column read is like-for-like. Appendix E2.
#   2. `spec_test_years`   — each spec's own window, for the test-year row that
#                            goes under Table E1's heading.
#   3. `lag_effect_matched`— the Figure 3 quantity recomputed on matched windows.
#   4. Figure `lag_effect_auroc_delta_matched.png`.
#
# WHAT IT DELIBERATELY EXCLUDES
# The two pre-crisis (RT2) columns test 2007 only, which is disjoint from
# 2011-2012. There is no shared window, so they cannot be matched at all and are
# omitted rather than silently misaligned. E1 keeps them, with the n=1 caveat it
# already carries.
#
# COST, TO BE STATED NOT HIDDEN: matching drops to n = 2 test years. That is low
# power. It is worth it here only because the window bias (~0.11 AUROC mean, up
# to 0.21) is several times the size of the lag effects being measured
# (~0.03-0.07) — an estimate whose bias exceeds its signal is worse than a
# noisier unbiased one. That trade is specific to this panel, not general.
#
# Needs cells 1-3 only (imports, BASE/OUTPUT, RESULT_FILES). Labels are defined
# locally on purpose: NB15 REDEFINES MODEL_LABELS_SHORT to short forms at cells 52/65/66,
# so inheriting it here would silently relabel the table.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple

COMMON_YEARS = [2011, 2012]          # the only window shared by all non-RT2 specs

_MODEL_ORDER = ['logistic_regression', 'decision_tree', 'random_forest',
                'xgboost', 'lightgbm', 'mlp']
# Labels match Table E1 in the manuscript exactly ('MLP-NN', not 'MLP') so E1 and
# E2 can be read row-for-row without the reader re-mapping names.
_MODEL_LABELS = {'logistic_regression': 'Logistic Regression',
                 'decision_tree': 'Decision Tree', 'random_forest': 'Random Forest',
                 'xgboost': 'XGBoost', 'lightgbm': 'LightGBM', 'mlp': 'MLP-NN'}
_DATASET_ORDER = ['micro', 'macro', 'integrated']
_DATASET_LABELS = {'micro': 'Micro', 'macro': 'Macro', 'integrated': 'Integrated'}
_DATASET_COLORS = {'micro': '#1f77b4', 'macro': '#ff7f0e', 'integrated': '#2ca02c'}

# (xlsx key, variant) per Appendix E column. Order mirrors E1 so the two tables
# can be read side by side; the two RT2 columns are absent by design (see above).
MATCHED_SPECS = {
    'Baseline / Lag-1':           ('11a_base_lag1',   'baseline_t1'),
    'Extended / Lag-1':           ('11b_robust_lag1', 'baseline_t1'),
    'Baseline / Lag-2':           ('11c_base_lag2',   'baseline_t1'),
    'Extended / Lag-2':           ('11d_robust_lag2', 'baseline_t1'),
    'Onset excl. / Lag-1':        ('11a_base_lag1',   'rt1'),
    'Onset excl. / Ext. Lag-1':   ('11b_robust_lag1', 'rt1'),
    'Onset excl. / Lag-2':        ('11c_base_lag2',   'rt1'),
    'Onset excl. / Ext. Lag-2':   ('11d_robust_lag2', 'rt1'),
}

# Every column of E1, including the two that cannot be matched — this drives the
# test-year row that goes under Table E1's heading.
ALL_E1_SPECS = dict(MATCHED_SPECS)
ALL_E1_SPECS['Pre-crisis / Lag-2']     = ('rt2a_base_lag2_fixed',   'rt2_2yr')
ALL_E1_SPECS['Pre-crisis / Ext. Lag-2'] = ('rt2b_robust_lag2_fixed', 'rt2_2yr')


def _all_results(spec_key, variant):
    """Per-year rows for one spec/variant. lr_no_dummies is a diagnostic model,
    excluded everywhere else in NB15, so excluded here too."""
    df = pd.read_excel(RESULT_FILES[spec_key], sheet_name='all_results')
    return df[(df['variant'] == variant) & (df['model'] != 'lr_no_dummies')]


def _mean_auroc(spec_key, variant, years=None):
    d = _all_results(spec_key, variant)
    if years is not None:
        d = d[d['test_year'].isin(years)]
    return d.groupby(['model', 'dataset'])['auroc'].mean()


# --- 1. the test-year row for Table E1 -------------------------------------
spec_test_years = pd.DataFrame([
    {'Specification': col,
     'Test years': ', '.join(str(y) for y in sorted(_all_results(k, v)['test_year'].unique())),
     'n': _all_results(k, v)['test_year'].nunique()}
    for col, (k, v) in ALL_E1_SPECS.items()
])

# --- 2. Appendix E2: every spec on the common window ------------------------
idx = pd.MultiIndex.from_product([_MODEL_ORDER, _DATASET_ORDER], names=['model', 'dataset'])
e2 = pd.DataFrame(index=idx)
for col, (k, v) in MATCHED_SPECS.items():
    e2[col] = _mean_auroc(k, v, COMMON_YEARS).reindex(idx)
e2 = e2.round(3)
e2.index = pd.MultiIndex.from_tuples(
    [(_MODEL_LABELS[m], _DATASET_LABELS[d]) for m, d in e2.index], names=['Model', 'Config'])

# --- 3. the Figure 3 quantity, matched --------------------------------------
# Held constant: indicator set. Varied: lag only. Both sides now on 2011-2012.
lag_rows = []
for set_label, (l1_key, l2_key) in [('Baseline indicators',  ('11a_base_lag1', '11c_base_lag2')),
                                    ('Extended indicators',  ('11b_robust_lag1', '11d_robust_lag2'))]:
    l1_un = _mean_auroc(l1_key, 'baseline_t1')                 # spec's own window
    l1_ma = _mean_auroc(l1_key, 'baseline_t1', COMMON_YEARS)   # matched
    l2    = _mean_auroc(l2_key, 'baseline_t1', COMMON_YEARS)   # already 2011-2012
    for (m, d) in idx:
        lag_rows.append({
            'Indicator set': set_label,
            'Model': _MODEL_LABELS[m], 'Config': _DATASET_LABELS[d],
            'Lag-1 (own window)': round(l1_un[(m, d)], 3),
            'Lag-1 (matched)':    round(l1_ma[(m, d)], 3),
            'Lag-2 (matched)':    round(l2[(m, d)], 3),
            'Delta unmatched':    round(l2[(m, d)] - l1_un[(m, d)], 3),
            'Delta matched':      round(l2[(m, d)] - l1_ma[(m, d)], 3),
        })
lag_effect_matched = pd.DataFrame(lag_rows)

# --- 4. matched version of Figure 3 -----------------------------------------
# Same grammar as the existing lag figure (3 rows, one per config; two bars per
# model = baseline vs extended indicator set) so the two are directly readable
# against each other.
fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharey=False)
width = 0.35
for ax, config in zip(axes, _DATASET_ORDER):
    color = _DATASET_COLORS[config]
    x = np.arange(len(_MODEL_ORDER))
    sub = lag_effect_matched[lag_effect_matched['Config'] == _DATASET_LABELS[config]]
    vals_base, vals_ext = [], []
    for mdl in _MODEL_ORDER:
        r = sub[sub['Model'] == _MODEL_LABELS[mdl]]
        vals_base.append(float(r[r['Indicator set'] == 'Baseline indicators']['Delta matched'].iloc[0]))
        vals_ext.append(float(r[r['Indicator set'] == 'Extended indicators']['Delta matched'].iloc[0]))
    b1 = ax.bar(x - width/2, vals_base, width, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    b2 = ax.bar(x + width/2, vals_ext,  width, color=color, alpha=0.4,  edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', lw=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels([_MODEL_LABELS[m] for m in _MODEL_ORDER], rotation=25, ha='right', fontsize=9)
    ax.set_title(_DATASET_LABELS[config], fontsize=10, fontweight='bold')
    ax.set_ylabel('ΔAUROC  (Lag-2 − Lag-1)', fontsize=9)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    for bar, v in zip([*b1, *b2], [*vals_base, *vals_ext]):
        if abs(v) >= 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, v + (0.003 if v >= 0 else -0.003),
                    f'{v:+.2f}', ha='center', va='bottom' if v >= 0 else 'top', fontsize=8)

pb = tuple(mpatches.Patch(color=_DATASET_COLORS[c], alpha=0.85) for c in _DATASET_ORDER)
pe = tuple(mpatches.Patch(color=_DATASET_COLORS[c], alpha=0.4)  for c in _DATASET_ORDER)
fig.legend(handles=[pb, pe],
           labels=['Baseline indicators  (lag-2 − lag-1)', 'Extended indicators  (lag-2 − lag-1)'],
           handler_map={tuple: HandlerTuple(ndivide=None, pad=0.1)},
           fontsize=9, loc='lower center', ncol=2, bbox_to_anchor=(0.5, 0.01))
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(OUTPUT / 'lag_effect_auroc_delta_matched.png', bbox_inches='tight', dpi=200)
plt.show()

# --- 5. export ---------------------------------------------------------------
# Its own file: results_15_robust.xlsx is rewritten wholesale by cell 27, so
# adding a sheet there would be overwritten on the next full run.
out = BASE / 'outputs/tables' / 'appendix_e_matched_window.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as xl:
    e2.to_excel(xl, sheet_name='E2_auroc_matched')
    spec_test_years.to_excel(xl, sheet_name='E1_test_years', index=False)
    lag_effect_matched.to_excel(xl, sheet_name='lag_effect_matched', index=False)

print(f"Saved → {out}")
print(f"Saved → lag_effect_auroc_delta_matched.png")
print("\n=== E1 test-year row ===")
print(spec_test_years.to_string(index=False))
print("\n=== Appendix E2: AUROC, all specifications on 2011-2012 ===")
print(e2.to_string())

_u = lag_effect_matched['Delta unmatched']
_m = lag_effect_matched['Delta matched']
print(f"\n=== lag effect: positive cells {int((_u > 0).sum())}/36 unmatched "
      f"→ {int((_m > 0).sum())}/36 matched ===")
print(lag_effect_matched.groupby('Indicator set')[['Delta unmatched', 'Delta matched']]
      .agg(['mean', lambda s: int((s > 0).sum())]).round(3).to_string())

## Appendix E figures — AUROC landscape and by-year views

**Figure E1** AUROC across specifications, pooled across models (matched windows).
**Figure E2** AUROC by test year, rows = indicator set, columns = model (lag-1 specs).

In [ ]:
# Appendix E figures — E1 (specification landscape) and E2 (by test year)
#   E1  auroc_landscape_pooled — AUROC across the 8 specifications, pooled across
#       models, one line per configuration, individual models faint behind.
#   E2  auroc_byyear_grid      — AUROC by test year; rows = indicator set,
#       columns = model, one line per configuration.
#
# WINDOWS — the two figures need different bases, for the reason the whole
# Appendix E split exists:
#   E1 reads ACROSS specifications, so it uses the MATCHED values (2011-2012),
#      i.e. Table E2's data. Own-window values would confound specification with
#      evaluation period: only the lag-1 specs include 2010, and 2010 is the weak
#      year (mean AUROC 0.477 against 0.764 and 0.799). The two pre-crisis specs
#      are omitted because they test 2007 alone and share no year with the rest.
#   E2 holds the specification fixed and varies the YEAR, so it uses the lag-1
#      specs, the only ones evaluated on all three years.
#
# WHAT IS NEVER POOLED. Configuration (micro/macro/integrated) is the object of
# the thesis, and indicator set carries Figure 2's finding that extension affects
# the configurations differently. Models are the only dimension pooled, and only
# in E1, where the faint lines keep the spread visible.
#
# CAVEAT CARRIED IN E1'S NOTE: the pooled line is an unweighted mean over six
# models whose levels differ a lot (DT ~0.65, LR ~0.95), so it summarises
# DIRECTION, not a typical AUROC.
#
# Self-contained on labels ON PURPOSE — NB15 redefines MODEL_LABELS_SHORT to short
# forms at cells 52/65/66, so inheriting it here would silently relabel panels.
# Needs cells 1-3 only (imports, BASE/OUTPUT, RESULT_FILES).
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

_B = BASE / 'outputs/results'
_SPECS = [('Baseline\nLag-1',          _B/'11a/results_11a_final.xlsx',        'baseline_t1'),
          ('Extended\nLag-1',          _B/'11b/results_11b_final.xlsx',        'baseline_t1'),
          ('Baseline\nLag-2',          _B/'11c/results_11c_draft4.xlsx',  'baseline_t1'),
          ('Extended\nLag-2',          _B/'11d/results_11d_final.xlsx',   'baseline_t1'),
          ('Onset excl.\nLag-1',       _B/'11a/results_11a_final.xlsx',        'rt1'),
          ('Onset excl. Ext.\nLag-1',  _B/'11b/results_11b_final.xlsx',        'rt1'),
          ('Onset excl.\nLag-2',       _B/'11c/results_11c_draft4.xlsx',  'rt1'),
          ('Onset excl. Ext.\nLag-2',  _B/'11d/results_11d_final.xlsx',   'rt1')]
_YEAR_SETS = [('Baseline indicators', _B/'11a/results_11a_final.xlsx'),
              ('Extended indicators', _B/'11b/results_11b_final.xlsx')]
_MATCH = [2011, 2012]; _YEARS = [2010, 2011, 2012]

_MO = ['logistic_regression','decision_tree','random_forest','xgboost','lightgbm','mlp']
_ML = {'logistic_regression':'Logistic Regression','decision_tree':'Decision Tree',
       'random_forest':'Random Forest','xgboost':'XGBoost','lightgbm':'LightGBM','mlp':'MLP-NN'}
_MS = {'logistic_regression':'LR','decision_tree':'DT','random_forest':'RF',
       'xgboost':'XGBoost','lightgbm':'LightGBM','mlp':'MLP-NN'}   # short: 6 narrow columns
_DO = ['micro','macro','integrated']
_DL = {'micro':'Micro','macro':'Macro','integrated':'Integrated'}
_DC = {'micro':'#1f77b4','macro':'#ff7f0e','integrated':'#2ca02c'}

def _rows(path, variant, years):
    df = pd.read_excel(path, sheet_name='all_results')
    return df[(df['variant']==variant) & (df['model']!='lr_no_dummies') & (df['test_year'].isin(years))]

# ---- Figure E1: specification landscape, pooled across models ---------------
labs = [s[0] for s in _SPECS]; x = np.arange(len(labs))
rec = []
for col, f, v in _SPECS:
    for (m, ds), g in _rows(f, v, _MATCH).groupby(['model','dataset']):
        rec.append({'spec':col,'model':m,'config':ds,'auroc':g['auroc'].mean()})
S = pd.DataFrame(rec)

fig, ax = plt.subplots(figsize=(11, 6.5))
for cfg in _DO:
    for mdl in _MO:                                          # faint = individual models
        s = S[(S.model==mdl)&(S.config==cfg)].set_index('spec')['auroc'].reindex(labs)
        ax.plot(x, s.values, color=_DC[cfg], alpha=0.18, lw=1.0, zorder=1)
    m = S[S.config==cfg].groupby('spec', observed=True)['auroc'].mean().reindex(labs)
    ax.plot(x, m.values, marker='o', ms=7, lw=3, color=_DC[cfg], label=_DL[cfg], zorder=3)
ax.axhline(0.5, color='grey', lw=0.9, ls=':')
ax.set_xticks(x); ax.set_xticklabels(labs, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('AUROC', fontsize=11); ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True); ax.set_ylim(0.4, 1.02)
ax.legend(fontsize=10, frameon=False, loc='lower right')
plt.tight_layout()
plt.savefig(OUTPUT/'auroc_landscape_pooled.png', bbox_inches='tight', dpi=200)
plt.show()

# ---- Figure E2: by test year; rows = indicator set, cols = model ------------
# Separating the indicator sets into ROWS rather than overlaying them as line
# styles keeps three lines per panel and makes baseline vs extended a direct
# vertical comparison per model.
xy = np.arange(len(_YEARS))
fig, axes = plt.subplots(2, 6, figsize=(17, 6.6), sharey=True, sharex=True)
for ri, (iset, f) in enumerate(_YEAR_SETS):
    d = _rows(f, 'baseline_t1', _YEARS)
    for ci, mdl in enumerate(_MO):
        ax = axes[ri, ci]
        for cfg in _DO:
            s = d[(d.model==mdl)&(d.dataset==cfg)].set_index('test_year')['auroc'].reindex(_YEARS)
            ax.plot(xy, s.values, marker='o', ms=5, lw=2.0, color=_DC[cfg], label=_DL[cfg])
        ax.axhline(0.5, color='grey', lw=0.8, ls=':')
        if ri == 0: ax.set_title(_MS[mdl], fontsize=11, fontweight='bold')
        ax.set_xticks(xy); ax.set_xticklabels(_YEARS, fontsize=8, rotation=45)
        ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
        ax.set_ylim(0.10, 1.05); ax.set_xlim(-0.25, 2.25)
    axes[ri, 0].set_ylabel(f'{iset}\n\nAUROC', fontsize=10, fontweight='bold')
for ax in axes[1, :]: ax.set_xlabel('Test year', fontsize=9)
h, l = axes[0,0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=3, fontsize=10, frameon=False, bbox_to_anchor=(0.5,-0.03))
plt.tight_layout(rect=[0,0.05,1,1])
plt.savefig(OUTPUT/'auroc_byyear_grid.png', bbox_inches='tight', dpi=200)
plt.show()

print('Saved → auroc_landscape_pooled.png (Figure E1), auroc_byyear_grid.png (Figure E2)')
print('\nFigure E1 — mean AUROC per configuration across specifications:')
print(S.groupby(['config','spec'], observed=True)['auroc'].mean().round(3).unstack().to_string())

## Headroom check (Discussion, D7)

Tests whether the uneven onset-exclusion gain is a ceiling effect. It is not.

In [ ]:
# Headroom check — is the uneven onset-exclusion gain a ceiling effect?
# D7 reports that the micro configuration gains least from excluding crisis-onset
# and immediate post-crisis years. The obvious objection is that micro simply had
# less room: it enters the baseline specifications as the strongest configuration,
# so the distance to a perfect AUROC of 1.0 is smaller.
#
# This cell tests that objection rather than assuming it. If room explained the
# smaller gain, each configuration would capture a similar FRACTION of the room
# available to it. Reported as:
#     room     = 1 - baseline AUROC          (distance left to perfect discrimination)
#     captured = (rt1 - baseline) / room     (share of that distance actually closed)
#
# Matched windows (2011-2012): rt1 cannot be scored on 2010, which has no crisis
# observations after the onset filter, so an unmatched comparison would confound
# the gain with the evaluation period.
#
# RESULT: the objection fails. Micro does have the least room, but captures only
# ~34% of it against ~65% for macro; and under the extended set, where micro has
# the MOST room, all three converge near half. Needs cells 1-3 only.
import pandas as pd, numpy as np

_HR_Y = [2011, 2012]

def _headroom(xlsx):
    df = pd.read_excel(xlsx, sheet_name='all_results')
    df = df[(df['model'] != 'lr_no_dummies') & (df['test_year'].isin(_HR_Y))]
    base = df[df['variant'] == 'baseline_t1'].groupby(['model', 'dataset'])['auroc'].mean()
    rt1  = df[df['variant'] == 'rt1'].groupby(['model', 'dataset'])['auroc'].mean()
    out = []
    for cfg in ['micro', 'macro', 'integrated']:
        keys = [k for k in base.index if k[1] == cfg and k in rt1.index]
        out.append({
            'Configuration':        cfg,
            'Baseline AUROC':       np.mean([base[k] for k in keys]),
            'Room left (1 - AUROC)': np.mean([1 - base[k] for k in keys]),
            '% of room captured':   np.mean([(rt1[k] - base[k]) / (1 - base[k]) for k in keys]) * 100,
        })
    return pd.DataFrame(out).round(3)

headroom_baseline = _headroom(RESULT_FILES['11a_base_lag1'])
headroom_extended = _headroom(RESULT_FILES['11b_robust_lag1'])

print('Headroom, BASELINE indicator set (matched test years 2011-2012)')
print(headroom_baseline.to_string(index=False))
print('\nHeadroom, EXTENDED indicator set (matched test years 2011-2012)')
print(headroom_extended.to_string(index=False))
print('\nRead: micro has the LEAST room in the baseline set yet captures the SMALLEST')
print('share of it, so the smaller gain is not a ceiling effect. Under the extended')
print('set micro has the MOST room and the three configurations converge near half.')